
# Medical Document OCR/Layout Workbench v9

This is an OCR-first notebook for the medical document extraction project.

It tests the refinements we discussed:

- stronger image OCR before parsing
- multiple preprocessing profiles
- word-box OCR, not only flat text
- visual line reconstruction
- OCR candidate scoring by layout quality
- OCR words / lines / candidates CSV outputs
- backend save gate remains separated from extraction status
- bad OCR goes to reupload/manual review instead of backend save

Run from the repository root after restarting the kernel.


In [1]:
from pathlib import Path
import os, re, json, html, hashlib, traceback, subprocess, unicodedata
from dataclasses import dataclass, field, asdict
from collections import defaultdict, Counter

import numpy as np
import pandas as pd

from PIL import Image, ImageOps, ImageEnhance, ImageFilter

try:
    import cv2
    CV2_AVAILABLE = True
except Exception:
    cv2 = None
    CV2_AVAILABLE = False

try:
    import fitz
    PYMUPDF_AVAILABLE = True
except Exception:
    fitz = None
    PYMUPDF_AVAILABLE = False

try:
    import pytesseract
    TESSERACT_AVAILABLE = True
except Exception:
    pytesseract = None
    TESSERACT_AVAILABLE = False

def find_repo_root(start=None):
    start = Path(start or os.getcwd()).resolve()
    for p in [start] + list(start.parents):
        if (p / ".git").exists() or (p / "samples").exists() or (p / "app").exists():
            return p
    return start

REPO_ROOT = find_repo_root()
SELECTED_DIR = REPO_ROOT / "samples" / "selected samples"
OUTPUT_DIR = REPO_ROOT / "notebook_outputs" / "selected_samples_v9_ocr_layout"
TEXT_DIR = OUTPUT_DIR / "chosen_texts"
SANITIZED_TEXT_DIR = OUTPUT_DIR / "sanitized_texts"
JSON_DIR = OUTPUT_DIR / "json"
PREVIEW_DIR = OUTPUT_DIR / "source_previews"
DEBUG_DIR = OUTPUT_DIR / "debug"
VARIANT_IMAGE_DIR = DEBUG_DIR / "variant_images"
VARIANT_TEXT_DIR = DEBUG_DIR / "variant_texts"

for d in [OUTPUT_DIR, TEXT_DIR, SANITIZED_TEXT_DIR, JSON_DIR, PREVIEW_DIR, DEBUG_DIR, VARIANT_IMAGE_DIR, VARIANT_TEXT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXTS = {".pdf", ".jpg", ".jpeg", ".png", ".webp", ".tif", ".tiff"}

def relpath(p):
    try:
        return str(Path(p).resolve().relative_to(REPO_ROOT))
    except Exception:
        return str(p)

print("Repo root:", REPO_ROOT)
print("Output dir:", OUTPUT_DIR)
print("Tesseract:", TESSERACT_AVAILABLE, "| OpenCV:", CV2_AVAILABLE, "| PyMuPDF:", PYMUPDF_AVAILABLE)


Repo root: /Users/aliebrahimi/Documents/Saman Salamat/Extract text from files/Extract-text-from-medical-documents
Output dir: /Users/aliebrahimi/Documents/Saman Salamat/Extract text from files/Extract-text-from-medical-documents/notebook_outputs/selected_samples_v9_ocr_layout
Tesseract: True | OpenCV: True | PyMuPDF: True


In [2]:
# =========================
# Git hygiene
# =========================

AUTO_UPDATE_GITIGNORE = True

RECOMMENDED_GITIGNORE_LINES = [
    "samples/selected samples/",
    "samples/selected_samples/",
    "notebook_outputs/",
    "debug_output/",
    "eval_storage/",
    "*.db",
]

def run_git(args):
    try:
        return subprocess.run(["git"] + args, cwd=REPO_ROOT, capture_output=True, text=True, check=False)
    except Exception:
        return None

def update_gitignore():
    gi = REPO_ROOT / ".gitignore"
    existing = gi.read_text(encoding="utf-8") if gi.exists() else ""
    missing_before = [x for x in RECOMMENDED_GITIGNORE_LINES if x not in existing]
    if AUTO_UPDATE_GITIGNORE and missing_before:
        with gi.open("a", encoding="utf-8") as f:
            f.write("\n# Medical extraction PHI/PII and notebook outputs\n")
            for x in missing_before:
                f.write(x + "\n")
    existing_after = gi.read_text(encoding="utf-8") if gi.exists() else ""
    missing_after = [x for x in RECOMMENDED_GITIGNORE_LINES if x not in existing_after]
    tracked = []
    res = run_git(["ls-files"])
    if res and res.returncode == 0:
        for x in res.stdout.splitlines():
            if x.startswith("samples/selected samples/") or x.startswith("samples/selected_samples/") or x.startswith("notebook_outputs/"):
                tracked.append(x)
    report = {
        "gitignore_path": relpath(gi),
        "missing_before": missing_before,
        "missing_after": missing_after,
        "tracked_sensitive_files_count": len(tracked),
        "tracked_sensitive_files_sample": tracked[:50],
        "cleanup_commands": [
            "git rm -r --cached 'samples/selected samples' 'samples/selected_samples' 'notebook_outputs' 2>/dev/null || true",
            "git add .gitignore",
            "git commit -m 'chore: ignore PHI samples and notebook outputs'",
        ],
    }
    (OUTPUT_DIR / "privacy_git_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    return report

print(json.dumps(update_gitignore(), ensure_ascii=False, indent=2))


{
  "gitignore_path": ".gitignore",
  "missing_before": [],
  "missing_after": [],
  "tracked_sensitive_files_count": 1513,
  "tracked_sensitive_files_sample": [
    "notebook_outputs/selected_samples_flow/json/0001_20260427_181919.json",
    "notebook_outputs/selected_samples_flow/json/0003_0014161672_14041209_O_00404121721.json",
    "notebook_outputs/selected_samples_flow/json/0005_20260427_181713.json",
    "notebook_outputs/selected_samples_flow/json/0006_0021858845_14041209_O_00404121731.json",
    "notebook_outputs/selected_samples_flow/json/0007_-2147483648_-210195.json",
    "notebook_outputs/selected_samples_flow/json/0008_0023471026_14041209_O_00404121728.json",
    "notebook_outputs/selected_samples_flow/json/0009_0025631314_14041209_O_00404121726.json",
    "notebook_outputs/selected_samples_flow/json/0010_-2147483648_-210193.json",
    "notebook_outputs/selected_samples_flow/json/0012_0024150010_14041209_O_00404121722.json",
    "notebook_outputs/selected_samples_flow/jso

In [3]:
# =========================
# Discover files
# =========================

ONLY_FILES = []  # optional: ["samples/selected samples/file.jpg"]

def discover_files():
    if ONLY_FILES:
        files = []
        for x in ONLY_FILES:
            p = Path(x)
            if not p.is_absolute():
                p = REPO_ROOT / p
            if p.exists() and p.is_file() and p.suffix.lower() in SUPPORTED_EXTS:
                files.append(p.resolve())
        return files

    folders = [SELECTED_DIR]
    found = []
    for folder in folders:
        if folder.exists():
            for ext in SUPPORTED_EXTS:
                found.extend(folder.rglob(f"*{ext}"))
                found.extend(folder.rglob(f"*{ext.upper()}"))
    seen, clean = set(), []
    for p in found:
        rp = p.resolve()
        if rp.is_file() and rp not in seen:
            seen.add(rp)
            clean.append(rp)
    return sorted(clean, key=lambda p: p.name)

FILES = discover_files()
print("Found", len(FILES), "files")
for i, p in enumerate(FILES, 1):
    print(f"{i:03d} | {p.name} | {p.suffix.lower()} | {p.stat().st_size/1024:.1f} KB | {relpath(p)}")

assert FILES, "No files found. Put files in samples/selected samples/ or set ONLY_FILES."


Found 20 files
001 | -2147483648_-210193.jpg | .jpg | 77.8 KB | samples/selected samples/-2147483648_-210193.jpg
002 | -2147483648_-210195.jpg | .jpg | 42.0 KB | samples/selected samples/-2147483648_-210195.jpg
003 | 0014161672_14041209_O_00404121721.pdf | .pdf | 44.2 KB | samples/selected samples/0014161672_14041209_O_00404121721.pdf
004 | 0020139871_14041209_O_00404121714.pdf | .pdf | 41.4 KB | samples/selected samples/0020139871_14041209_O_00404121714.pdf
005 | 0021858845_14041209_O_00404121731.pdf | .pdf | 42.9 KB | samples/selected samples/0021858845_14041209_O_00404121731.pdf
006 | 0023471026_14041209_O_00404121728.pdf | .pdf | 40.8 KB | samples/selected samples/0023471026_14041209_O_00404121728.pdf
007 | 0024150010_14041209_O_00404121722.pdf | .pdf | 44.1 KB | samples/selected samples/0024150010_14041209_O_00404121722.pdf
008 | 0025631314_14041209_O_00404121726.pdf | .pdf | 41.2 KB | samples/selected samples/0025631314_14041209_O_00404121726.pdf
009 | 0025692283_14041209_O_00404

In [4]:
# =========================
# Config, aliases, normalization
# =========================

CONFIG = {
    "max_pdf_pages": 5,
    "min_ocr_text_len": 20,
    "psm_modes": [4, 6, 11, 12],
    "tesseract_langs": ["eng+fas", "eng"],
    "run_rotation_variants": True,
    "save_variant_images": True,
    "save_variant_texts": True,
    "min_good_layout_score": 0.62,
    "min_usable_layout_score": 0.38,
    "backend_patient_context_available": False,
    "require_report_date_for_backend_save": True,
    "require_patient_or_backend_context_for_backend_save": True,
}

MEDICAL_KEYWORDS = [
    "laboratory","lab","hematology","biochemistry","serology","hormone","thyroid","urine","culture",
    "cbc","wbc","rbc","hemoglobin","platelet","fbs","glucose","creatinine","cholesterol","triglycerides",
    "tsh","ferritin","vitamin","crp","pt","inr","ptt","reference range","method",
    "آزمایشگاه","آزمايشگاه","پاتوبیولوژی","پاتوبيولوژی","درمانگاه","بیمارستان","پذیرش","نتیجه","ادرار","خون","نام بیمار",
]

LAB_ALIASES = {
    "WBC": ["WBC","W.B.C","White Blood Cell"],
    "RBC": ["RBC","R.B.C","Red Blood Cell"],
    "HGB": ["HGB","Hb","Hemoglobin","Haemoglobin","Hemoglobine"],
    "HCT": ["HCT","H.C.T","Hematocrit","Het"],
    "MCV": ["MCV","M.C.V"],
    "MCH": ["MCH","M.C.H"],
    "MCHC": ["MCHC","M.C.H.C"],
    "RDW-CV": ["RDW-CV","RDW CV","RDW."],
    "RDW-SD": ["RDW-SD","RDW SD"],
    "PLT": ["PLT","Platelet","Platelets"],
    "PDW": ["PDW"],
    "MPV": ["MPV"],
    "NEUT%": ["NEUT%","Neut%","Neutrophils","Neutrophil"],
    "LYMPH%": ["LYMPH%","Lymph%","Lymphocytes","Lymphocyte","Lympho"],
    "MONO%": ["MONO%","Mono%","Monocytes","Monocyte"],
    "EO%": ["EO%","EOS%","Eosinophils","Eosinophil"],
    "BASO%": ["BASO%","Basophils","Basophil"],
    "ESR": ["ESR","Erythrocyte Sediment Rate","Erythrocyte Sedimentation Rate"],
    "FBS": ["Fasting Blood Glucose","Fasting blood sugar","Fasting blood segar","FBS","Glucose","Blood sugar"],
    "HbA1c": ["HbA1c","Hemoglobin A1c","Hemoglobine A1c","Hemoglobin Ate"],
    "EAG": ["Estimated Average Glucose","EAG","EAS"],
    "BUN": ["BUN","Blood Urea Nitrogen","Blood Urea nitrogen"],
    "Urea": ["Urea"],
    "Creatinine": ["Creatinine","Creatininee","Creatinin"],
    "Uric Acid": ["Uric Acid"],
    "Total Cholesterol": ["Total Cholesterol","Cholesterol","Cholestrol"],
    "Triglycerides": ["Triglycerides","Triglycerideses","Triglycerid","TG"],
    "HDL": ["HDL","HDL Cholesterol","HDL-Cholesterol"],
    "LDL": ["LDL","LDL Cholesterol","LDL-Cholesterol","LDL - Cholesterol"],
    "AST": ["AST","SGOT","SGOT(AST)","AST(SGOT)"],
    "ALT": ["ALT","SGPT","SGPT(ALT)"],
    "TSH": ["TSH","Thyroid Stimulating Hormone"],
    "T3": ["T3","Tri iodothyronine","Triiodothyronine"],
    "T4": ["T4","Thyroxine"],
    "Free T4": ["Free T4","FT4"],
    "Vitamin D": ["Vitamin D","25 Hydroxy Vitamin D","25-Hydroxy Vitamin D3","25-OH Vitamin D3"],
    "Ferritin": ["Ferritin","Ferrin"],
    "CRP": ["CRP","C-Reactive protein"],
    "PT": ["PT","Prothrombin Time"],
    "PT Control": ["PT Control","PT Controt"],
    "INR": ["INR"],
    "PTT": ["PTT","Activated PTT","Activated PT"],
    "Urine Culture": ["Urine Culture","Urine Cultures","Culture"],
    "Color": ["Color","olor"],
    "Appearance": ["Appearance","Appenrance"],
    "Specific Gravity": ["Specific Gravity","Sp. Gravity","Specie Gravity"],
    "pH": ["pH","PH"],
    "Protein": ["Protein"],
    "Urine Glucose": ["Urine Glucose","Glucose"],
    "Ketone": ["Ketone","Kewoe"],
    "Bilirubin Urine": ["Bilirubin","Darwin"],
    "Urobilinogen": ["Urobilinogen","Uobinogen"],
    "Nitrite": ["Nitrite","Nirive"],
    "Blood/Hb": ["Blood/Hb","Hemoglobin","Blood"],
    "WBC/HPF": ["WBC/HPF","W.B.C / HPF"],
    "RBC/HPF": ["RBC/HPF","R.B.C / HPF"],
    "Bacteria": ["Bacteria","Dacterta","acterta"],
    "Mucus": ["Mucus"],
    "Casts": ["Casts","Cast"],
    "Crystals": ["Crystals","Crystal"],
}

ALL_ALIAS_PHRASES = [(std, alias) for std, aliases in LAB_ALIASES.items() for alias in aliases]
ALL_ALIAS_LOWER = [(std, alias, alias.lower()) for std, alias in ALL_ALIAS_PHRASES]

ROW_SANITY_LIMITS = {
    "WBC": (0.5,80), "RBC": (1.0,8.0), "HGB": (3.0,25.0), "HCT": (10.0,75.0),
    "MCV": (50.0,130.0), "MCH": (15.0,45.0), "MCHC": (20.0,45.0), "PLT": (10.0,1500.0),
    "HbA1c": (3.0,20.0), "FBS": (20.0,600.0), "Creatinine": (0.1,20.0),
    "Ferritin": (1.0,3000.0), "Vitamin D": (1.0,200.0), "TSH": (0.001,200.0),
}

QUALITATIVE_VALID_VALUES = {
    "negative","positive","trace","few","rare","many","not seen","clear","yellow","amber","red","pale","straw","normal",
    "turbid","cloudy","semi-clear","acid","alkaline","look at culture","no growth after 24 hrs","no growth after 48 hrs",
    "no growth after 24 hours","no growth after 48 hours","no bacteria growth after 48 hrs",
}
QUALITATIVE_TESTS = {
    "Urine Culture","Color","Appearance","Protein","Urine Glucose","Ketone","Bilirubin Urine","Urobilinogen",
    "Nitrite","Blood/Hb","Bacteria","Mucus","Casts","Crystals",
}

DIGIT_MAP = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")

def normalize_digits(text):
    return (text or "").translate(DIGIT_MAP)

def normalize_text(text):
    text = unicodedata.normalize("NFKC", text or "")
    text = normalize_digits(text)
    text = text.replace("ي","ی").replace("ك","ک").replace("ۀ","ه")
    corrections = {
        "خائم":"خانم","خانمز":"خانم ز","خانمزهرا":"خانم زهرا","خانمسیده":"خانم سیده","خانمفرزانه":"خانم فرزانه",
        "بیمار نام":"نام بیمار","بيمار نام":"نام بیمار","پزشک نام":"نام پزشک","پزشك نام":"نام پزشک",
        "پذیرش تاریخ":"تاریخ پذیرش","پذيرش تاریخ":"تاریخ پذیرش","پدیرش":"پذیرش","پذبرش":"پذیرش",
        "پذيرش":"پذیرش","تاريخ":"تاریخ","دكتر":"دکتر","پزشك":"پزشک",
        "Fasting Blood Glucase":"Fasting Blood Glucose","Fasting blood segar":"Fasting blood sugar",
        "Hemoglobine Alc":"Hemoglobin A1c","Hemoglobin Ate":"Hemoglobin A1c","Cholestrol":"Cholesterol",
        "Triglycerideses":"Triglycerides","Creatininee":"Creatinine","Creatinin":"Creatinine",
    }
    for a,b in corrections.items():
        text = text.replace(a,b)
    text = re.sub(r"[^\S\n]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def clean_value(v, max_len=120):
    if not v:
        return None
    v = normalize_text(v)
    v = re.sub(r"^[\s:：|#*/\\.,؛;+_-]+|[\s:：|#*/\\.,؛;+_-]+$", "", v).strip()
    v = re.sub(r"\s+", " ", v)
    if not v or len(v) > max_len:
        return None
    return v

def sanitize_text(text):
    t = text or ""
    t = re.sub(r"(?<!\d)\d{10}(?!\d)", "**********", t)
    t = re.sub(r"(?<!\d)09\d{9}(?!\d)", "09*********", t)
    return t

def safe_id(path):
    stem = re.sub(r"[^\w\u0600-\u06FF.-]+", "_", Path(path).stem)
    return stem[:80] or hashlib.md5(str(path).encode()).hexdigest()[:10]

def hash_value(v):
    return hashlib.sha256(str(v).encode("utf-8")).hexdigest() if v else None

def validate_jalali_date(date):
    if not date:
        return False
    date = normalize_digits(date)
    m = re.match(r"^(13[9][0-9]|14[0-1][0-9]|1420)[/-](\d{1,2})[/-](\d{1,2})$", date)
    if not m:
        return False
    y, mo, d = int(m.group(1)), int(m.group(2)), int(m.group(3))
    return 1390 <= y <= 1420 and 1 <= mo <= 12 and 1 <= d <= 31

def file_mime_guess(path):
    return {".pdf":"application/pdf",".jpg":"image/jpeg",".jpeg":"image/jpeg",".png":"image/png",".webp":"image/webp",".tif":"image/tiff",".tiff":"image/tiff"}.get(Path(path).suffix.lower(), "application/octet-stream")


In [5]:
# =========================
# Image quality, preview, preprocessing variants
# =========================

def assess_image_quality(path):
    try:
        img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
        w,h = img.size
        gray = np.array(img.convert("L"))
        mean = float(np.mean(gray)); std = float(np.std(gray))
        lap = float(cv2.Laplacian(gray, cv2.CV_64F).var()) if CV2_AVAILABLE else 0.0
        blur_score = min(1.0, lap/500.0) if lap else 0.5
        brightness_score = max(0.0, 1 - abs(mean-127)/127)
        contrast_score = min(1.0, std/80.0)
        resolution_score = min(1.0, (w*h)/(1000*1000))
        overall = blur_score*.35 + brightness_score*.20 + contrast_score*.25 + resolution_score*.20
        issues = []
        if lap and blur_score < .18: issues.append("severe_blur")
        if contrast_score < .25: issues.append("low_contrast")
        if mean < 45: issues.append("too_dark")
        if mean > 220: issues.append("too_bright")
        if resolution_score < .10: issues.append("low_resolution")
        if h > w*1.35 or w > h*1.35: issues.append("possible_rotation_or_sideways")
        status = "good_quality" if overall >= 0.60 else ("needs_preprocessing" if overall >= 0.22 else "poor_quality")
        return {"status":status,"overall_quality_score":round(overall,3),"issues":issues,"width":w,"height":h,
                "brightness_mean":round(mean,2),"contrast_std":round(std,2),"laplacian_variance":round(lap,2)}
    except Exception as e:
        return {"status":"unreadable","overall_quality_score":0.0,"issues":["unreadable_image",str(e)]}

def assess_file_quality(path):
    if Path(path).suffix.lower() == ".pdf":
        return {"status":"pdf_quality_not_assessed_here","overall_quality_score":None,"issues":[]}
    return assess_image_quality(path)

def create_source_preview(path, sid):
    out = PREVIEW_DIR / f"{sid}.png"
    try:
        if Path(path).suffix.lower() == ".pdf" and PYMUPDF_AVAILABLE:
            doc = fitz.open(path); page = doc[0]
            pix = page.get_pixmap(matrix=fitz.Matrix(2,2), alpha=False)
            pix.save(str(out)); doc.close()
        else:
            img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
            img.thumbnail((900,900)); img.save(out)
        return relpath(out)
    except Exception:
        return None

def pil_clahe(img, clip=2.0):
    if not CV2_AVAILABLE:
        return ImageOps.autocontrast(img.convert("L")).convert("RGB")
    gray = np.array(img.convert("L"))
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8,8))
    return Image.fromarray(clahe.apply(gray)).convert("RGB")

def pil_shadow_remove(img):
    if not CV2_AVAILABLE:
        return ImageOps.autocontrast(img.convert("L")).convert("RGB")
    arr = np.array(img.convert("RGB"))
    planes = cv2.split(arr)
    result_planes = []
    for plane in planes:
        dilated = cv2.dilate(plane, np.ones((7,7), np.uint8))
        bg = cv2.medianBlur(dilated, 21)
        diff = 255 - cv2.absdiff(plane, bg)
        norm = cv2.normalize(diff, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX)
        result_planes.append(norm)
    return Image.fromarray(cv2.merge(result_planes)).convert("RGB")

def image_preprocess_variants(path, sid):
    base = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    variants = []
    def add(name, im):
        variants.append((name, im.convert("RGB")))
    add("original_oriented", base)
    add("upscale2_original", base.resize((base.width*2, base.height*2), Image.Resampling.LANCZOS))
    if max(base.size) < 1500:
        add("upscale3_original", base.resize((base.width*3, base.height*3), Image.Resampling.LANCZOS))
    gray = ImageOps.grayscale(base)
    add("gray_autocontrast", ImageOps.autocontrast(gray))
    add("gray_autocontrast_upscale2", ImageOps.autocontrast(gray).resize((base.width*2, base.height*2), Image.Resampling.LANCZOS))
    add("mild_contrast", ImageEnhance.Contrast(gray).enhance(1.4))
    add("sharpen_light", ImageOps.autocontrast(gray.filter(ImageFilter.SHARPEN)))
    add("clahe_light", pil_clahe(base, 1.5))
    add("clahe_medium", pil_clahe(base, 2.5))
    add("shadow_removed", pil_shadow_remove(base))
    if CV2_AVAILABLE:
        arr = np.array(gray)
        thr = cv2.adaptiveThreshold(arr,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY,31,11)
        add("threshold_fallback", Image.fromarray(thr))
    if CONFIG["run_rotation_variants"]:
        for name, im in list(variants):
            if name in {"original_oriented","gray_autocontrast","gray_autocontrast_upscale2","clahe_light","shadow_removed"}:
                for deg in [90,180,270]:
                    add(f"{name}_rot{deg}", im.rotate(deg, expand=True))
    if CONFIG["save_variant_images"]:
        folder = VARIANT_IMAGE_DIR / sid
        folder.mkdir(parents=True, exist_ok=True)
        for name, im in variants:
            im.save(folder / f"{name}.png")
    return variants


In [6]:
# =========================
# OCR word boxes, visual lines, scoring
# =========================

@dataclass
class OCRCandidate:
    candidate_id: str
    page_number: int
    variant_name: str
    psm: int | None
    lang: str
    text: str
    confidence: float
    score: float
    status: str
    score_details: dict
    words: list = field(default_factory=list)
    lines: list = field(default_factory=list)
    error: str | None = None

def run_tesseract_to_words(img, lang, psm):
    if not TESSERACT_AVAILABLE:
        return [], 0.0, "pytesseract_not_available"
    cfg = f"--psm {psm} -c preserve_interword_spaces=1"
    try:
        data = pytesseract.image_to_data(img, lang=lang, config=cfg, output_type=pytesseract.Output.DICT)
        words = []
        n = len(data.get("text", []))
        for i in range(n):
            txt = str(data["text"][i] or "").strip()
            if not txt:
                continue
            try:
                conf = float(data.get("conf", ["-1"])[i])
            except Exception:
                conf = -1
            if conf < 0:
                continue
            words.append({
                "text": normalize_text(txt),
                "conf": conf/100.0,
                "left": int(data.get("left",[0]*n)[i]),
                "top": int(data.get("top",[0]*n)[i]),
                "width": int(data.get("width",[0]*n)[i]),
                "height": int(data.get("height",[0]*n)[i]),
                "block_num": int(data.get("block_num",[0]*n)[i]),
                "par_num": int(data.get("par_num",[0]*n)[i]),
                "line_num": int(data.get("line_num",[0]*n)[i]),
                "word_num": int(data.get("word_num",[0]*n)[i]),
            })
        conf = float(np.mean([w["conf"] for w in words])) if words else 0.0
        return words, conf, None
    except Exception as e:
        return [], 0.0, str(e)

def reconstruct_visual_lines(words):
    if not words:
        return []
    groups = defaultdict(list)
    for w in words:
        groups[(w["block_num"], w["par_num"], w["line_num"])].append(w)
    lines = []
    for key, items in groups.items():
        items = sorted(items, key=lambda x: x["left"])
        text = " ".join(w["text"] for w in items).strip()
        if not text:
            continue
        left = min(w["left"] for w in items)
        top = min(w["top"] for w in items)
        right = max(w["left"] + w["width"] for w in items)
        bottom = max(w["top"] + w["height"] for w in items)
        lines.append({
            "line_id": 0, "text": normalize_text(text),
            "left": left, "top": top, "right": right, "bottom": bottom,
            "width": right-left, "height": bottom-top,
            "word_count": len(items),
            "avg_conf": float(np.mean([w["conf"] for w in items])),
        })
    lines = sorted(lines, key=lambda x: (x["top"], x["left"]))
    for i, line in enumerate(lines, 1):
        line["line_id"] = i
    return lines

def text_from_lines(lines):
    return "\n".join(l["text"] for l in lines if l.get("text"))

def alias_hits(text):
    low = normalize_text(text).lower()
    return [(std, alias) for std, alias, alow in ALL_ALIAS_LOWER if alow and alow in low]

def medical_hits(text):
    low = normalize_text(text).lower()
    return [k for k in MEDICAL_KEYWORDS if normalize_text(k).lower() in low]

def line_table_pattern_count(lines):
    count = 0
    for l in lines:
        txt = l["text"]
        has_alias = bool(alias_hits(txt))
        nums = re.findall(r"(?<!\d)[<>*]?\d+(?:\.\d+)?(?!\d)", txt)
        unit = re.search(r"(mg/dL|mg/dl|g/dL|U/L|%|fL|pg|10\^|ng/mL|uIU/mL)", txt, re.I)
        if has_alias and (nums or unit):
            count += 1
        elif len(nums) >= 2 and unit:
            count += 1
    return count

def common_field_signal_count(text):
    signals = [r"نام\s*بیمار", r"نام\s*مراجعه", r"کد\s*ملی", r"تاریخ\s*پذیرش", r"Report\s*Date", r"Patient\s*Name", r"سن", r"پذیرش"]
    return sum(1 for s in signals if re.search(s, text, re.I))

def impossible_value_penalty(text):
    penalties = 0
    for pat in [r"\bMCH\b[^\n]{0,40}\b\d{3,}\b", r"HbA1c[^\n]{0,40}\b\d{3,}\b", r"\bWBC\b[^\n]{0,40}\b[5-9]\d\b"]:
        if re.search(pat, text, re.I):
            penalties += 1
    return penalties

def gibberish_ratio(text):
    toks = re.findall(r"[A-Za-z\u0600-\u06FF0-9]+", text or "")
    if not toks:
        return 1.0
    random_short = sum(1 for t in toks if len(t) <= 2 and not t.isdigit())
    long_gib = sum(1 for t in toks if len(t) >= 12 and not re.search(r"[aeiouآایو]", t, re.I))
    return min(1.0, (random_short*0.25 + long_gib) / len(toks))

def culture_signal(text):
    return bool(re.search(r"No\s+(?:bacteria\s+)?growth\s+after\s+(?:24|48)\s*(?:hrs?|hours)\.?", normalize_text(text), re.I))

def score_ocr_candidate(text, lines, confidence):
    text = normalize_text(text)
    ah = alias_hits(text)
    mh = medical_hits(text)
    nums = re.findall(r"(?<!\d)[<>*]?\d+(?:\.\d+)?(?!\d)", text)
    table_patterns = line_table_pattern_count(lines)
    common_signals = common_field_signal_count(text)
    gib = gibberish_ratio(text)
    culture = culture_signal(text)
    impossible = impossible_value_penalty(text)
    line_count = len([l for l in lines if l.get("text")])
    avg_line_conf = float(np.mean([l["avg_conf"] for l in lines])) if lines else 0.0

    score = (
        min(len(text),3500)/3500*0.10 +
        min(len(ah),30)/30*0.23 +
        min(len(mh),18)/18*0.14 +
        min(len(nums),100)/100*0.10 +
        min(table_patterns,20)/20*0.18 +
        min(common_signals,8)/8*0.09 +
        min(line_count,60)/60*0.06 +
        max(confidence, avg_line_conf)*0.16 -
        gib*0.20 -
        min(impossible,5)/5*0.12
    )
    if culture:
        score = max(score, 0.42)
    score = max(0.0, min(1.0, float(score)))
    if len(text.strip()) < CONFIG["min_ocr_text_len"]:
        status = "poor_ocr_text"
    elif culture and len(text.strip()) >= 40:
        status = "usable_short_culture_text"
    elif score >= CONFIG["min_good_layout_score"]:
        status = "good_layout_text"
    elif score >= CONFIG["min_usable_layout_score"]:
        status = "usable_noisy_layout_text"
    else:
        status = "gibberish_or_bad_layout_text"
    details = {
        "alias_hits": len(ah),
        "medical_hits": len(mh),
        "numeric_hits": len(nums),
        "table_patterns": table_patterns,
        "common_field_signals": common_signals,
        "line_count": line_count,
        "gibberish_ratio": round(gib,3),
        "culture_signal": culture,
        "impossible_value_penalty": impossible,
        "avg_word_confidence": round(confidence,3),
        "avg_line_confidence": round(avg_line_conf,3),
    }
    return score, status, details

def ocr_image_candidate(img, page_number, variant_name, psm, lang, candidate_id):
    words, conf, err = run_tesseract_to_words(img, lang, psm)
    lines = reconstruct_visual_lines(words)
    text = text_from_lines(lines)
    score, status, details = score_ocr_candidate(text, lines, conf)
    return OCRCandidate(candidate_id, page_number, variant_name, psm, lang, text, conf, score, status, details, words, lines, err)


In [7]:
# =========================
# PDF/Image OCR candidates
# =========================

def extract_pdf_text_layer(path):
    if not PYMUPDF_AVAILABLE:
        return "", [], {"error": "pymupdf_not_available"}
    try:
        doc = fitz.open(path)
        pages = []
        for page in doc[:CONFIG["max_pdf_pages"]]:
            pages.append(normalize_text(page.get_text() or ""))
        meta = {"page_count": doc.page_count, "used_pages": len(pages), "pdf_type": "text_pdf" if any(p.strip() for p in pages) else "scanned_pdf"}
        doc.close()
        return "\f".join(pages), pages, meta
    except Exception as e:
        return "", [], {"error": str(e)}

def render_pdf_pages(path, sid):
    if not PYMUPDF_AVAILABLE:
        return []
    out_dir = DEBUG_DIR / "pdf_pages" / sid
    out_dir.mkdir(parents=True, exist_ok=True)
    out = []
    doc = fitz.open(path)
    for i, page in enumerate(doc[:CONFIG["max_pdf_pages"]], 1):
        pix = page.get_pixmap(matrix=fitz.Matrix(2,2), alpha=False)
        p = out_dir / f"page_{i:03d}.png"
        pix.save(str(p))
        out.append(p)
    doc.close()
    return out

def ocr_candidates_for_image_path(path, sid, page_number=1):
    cands = []
    for variant_name, img in image_preprocess_variants(path, f"{sid}_page{page_number}"):
        for psm in CONFIG["psm_modes"]:
            for lang in CONFIG["tesseract_langs"]:
                cid = f"{sid}_p{page_number}_{variant_name}_psm{psm}_{lang}"
                cands.append(ocr_image_candidate(img, page_number, variant_name, psm, lang, cid))
    if CONFIG["save_variant_texts"]:
        folder = VARIANT_TEXT_DIR / sid
        folder.mkdir(parents=True, exist_ok=True)
        for i, c in enumerate(sorted(cands, key=lambda x: x.score, reverse=True)[:30]):
            (folder / f"{i:02d}_{c.variant_name}_psm{c.psm}_{c.lang}_score{c.score:.3f}.txt").write_text(c.text or "", encoding="utf-8")
    return cands

def get_ocr_candidates_for_file(path, sid):
    path = Path(path)
    if path.suffix.lower() == ".pdf":
        text, pages, meta = extract_pdf_text_layer(path)
        if len(text.strip()) >= CONFIG["min_ocr_text_len"]:
            lines = [{"line_id":i+1,"text":l,"left":None,"top":i,"right":None,"bottom":i,"width":None,"height":None,"word_count":len(l.split()),"avg_conf":0.95}
                     for i,l in enumerate(text.splitlines()) if l.strip()]
            score, status, details = score_ocr_candidate(text, lines, 0.95)
            cand = OCRCandidate(f"{sid}_pdf_text_layer", 1, "pdf_text_layer", None, "embedded", text, 0.95, max(score,0.90), "good_layout_text", {**details, "pdf_meta":meta}, [], lines, None)
            return [cand], meta
        all_cands = []
        for i, img_path in enumerate(render_pdf_pages(path, sid), 1):
            all_cands += ocr_candidates_for_image_path(img_path, sid, i)
        return all_cands, meta
    return ocr_candidates_for_image_path(path, sid, 1), {}

def select_best_candidate(candidates):
    if not candidates:
        return OCRCandidate("empty", 1, "none", None, "none", "", 0, 0, "empty_text", {}, [], [], "no_candidates")
    return sorted(candidates, key=lambda c: (c.score, c.confidence, len(c.text)), reverse=True)[0]


In [8]:
# =========================
# Sections and OCR output rows
# =========================

SECTION_KEYWORDS = {
    "hematology": ["Hematology","CBC","Complete Blood Count","WBC","RBC","Hemoglobin"],
    "biochemistry": ["Biochemistry","Fasting Blood","Glucose","Urea","Creatinine","Cholesterol"],
    "hormone": ["Hormone","Immunoassays","Endocrinology","TSH","Free T4"],
    "urine": ["Urine Analysis","Urine analysts","Macroscopic","Microscopic","Color","Appearance"],
    "culture": ["Culture","Sensitivity","No growth","No bacteria growth"],
}

def detect_sections(lines):
    sections = []
    current = "unknown"
    start_line = 1
    for line in lines:
        low = line["text"].lower()
        detected = None
        for sec, keys in SECTION_KEYWORDS.items():
            if any(k.lower() in low for k in keys):
                detected = sec
                break
        if detected and detected != current:
            if current != "unknown":
                sections.append({"section":current,"start_line":start_line,"end_line":line["line_id"]-1})
            current = detected
            start_line = line["line_id"]
    if lines:
        sections.append({"section":current,"start_line":start_line,"end_line":lines[-1]["line_id"]})
    return sections

def line_to_section(line_id, sections):
    for s in sections:
        if s["start_line"] <= line_id <= s["end_line"]:
            return s["section"]
    return "unknown"

def make_words_lines_sections_tables(doc_index, filename, candidate, source_rel):
    words_rows = []
    for wi, w in enumerate(candidate.words, 1):
        row = dict(w)
        row.update({"index":doc_index,"filename":filename,"candidate_id":candidate.candidate_id,
                    "page_number":candidate.page_number,"variant_name":candidate.variant_name,
                    "psm":candidate.psm,"lang":candidate.lang,"word_index":wi,"source_relative_path":source_rel})
        words_rows.append(row)

    sections = detect_sections(candidate.lines)
    line_rows = []
    for l in candidate.lines:
        row = dict(l)
        row.update({"index":doc_index,"filename":filename,"candidate_id":candidate.candidate_id,
                    "page_number":candidate.page_number,"variant_name":candidate.variant_name,
                    "psm":candidate.psm,"lang":candidate.lang,"section":line_to_section(l["line_id"], sections),
                    "source_relative_path":source_rel})
        line_rows.append(row)

    section_rows = []
    for s in sections:
        row = dict(s)
        row.update({"index":doc_index,"filename":filename,"candidate_id":candidate.candidate_id,"source_relative_path":source_rel})
        section_rows.append(row)
    return words_rows, line_rows, section_rows


In [9]:
# =========================
# Classification and common fields
# =========================

TABLE_TOKENS_RE = re.compile(r"\b(Result|Unit|Reference|Range|WBC|RBC|FBS|TSH|CBC|Biochemistry|Hematology|Platelet|Creatinine|Glucose|Method)\b", re.I)
PATIENT_LABEL_RE = re.compile(r"(نام\s*بیمار|نام\s*مراجعه|Patient\s*Name)", re.I)
DOCTOR_LABEL_RE = re.compile(r"(نام\s*پزشک|پزشک\s*معالج|Doctor|Physician)", re.I)

def is_medical_text(text):
    low = normalize_text(text).lower()
    hits = [k for k in MEDICAL_KEYWORDS if normalize_text(k).lower() in low]
    score = min(1.0, len(hits)/4)
    return score >= 0.15, hits, score

def classify_document(text):
    low = normalize_text(text).lower()
    signals = {
        "lab": ["laboratory","lab","hematology","biochemistry","serology","urine","culture","cbc","wbc","rbc","fbs","glucose","tsh","آزمایشگاه","آزمايشگاه","درمانگاه"],
        "radiology": ["radiology","ultrasound","sonography","mri","ct","x-ray","findings","impression","سونوگرافی","سونوگرافي","رادیولوژی"],
        "pap_smear": ["pap smear","hpv","nilm","ascus","asc-us","lsil","hsil","پاپ اسمیر"],
    }
    best = ("unknown_medical", 0.0, [])
    for dt, sigs in signals.items():
        found = [s for s in sigs if normalize_text(s).lower() in low]
        if len(found) > len(best[2]):
            best = (dt, min(1.0, len(found)/5), found)
    return best

def reject_name_candidate(v):
    if not v: return True
    if len(v) > 90: return True
    if TABLE_TOKENS_RE.search(v): return True
    if len(re.findall(r"\d", v)) > 2: return True
    if re.search(r"پزشک|دکتر|Doctor|Result|Reference|Unit|Method|تاریخ|پذیرش|آزمایشگاه|Laboratory", v, re.I): return True
    return False

def reject_doctor_candidate(v):
    if not v: return True
    if len(v) > 90 or len(v) < 3: return True
    if PATIENT_LABEL_RE.search(v): return True
    if TABLE_TOKENS_RE.search(v): return True
    if re.search(r"آزمایشگاه|Laboratory|پزشکی آزمایشگاه|نام بیمار|بیمار|پذیرش|تاریخ|سن", v, re.I): return True
    if re.search(r"دکتر\s*[0-9]{1,3}", v): return True
    return False

def compact_name_spacing(v):
    v = normalize_text(v)
    v = re.sub(r"(خانم|آقای|آقاي)(?=[آ-ی])", r"\1 ", v)
    return re.sub(r"\s+", " ", v).strip()

def extract_strict_national_id(full):
    m = re.search(r"(?:کد\s*ملی|کد\s*ملي|كد\s*ملی|كد\s*ملي|National\s*ID|National\s*Code)\s*[:：]?\s*([0-9]{10})", full, re.I)
    if m:
        return m.group(1), m.group(0)
    return None, None

def extract_date(full):
    patterns = [
        r"(?:تاریخ\s*پذیرش|تاریخ\s*جواب|تاریخ\s*گزارش|Report\s*Date|Date)\s*[:：]?\s*(?:\d{1,2}:\d{2}:\d{2}\s*[-–]?\s*)?([0-9]{4}[/-][0-9]{1,2}[/-][0-9]{1,2})",
        r"([0-9]{4}[/-][0-9]{1,2}[/-][0-9]{1,2})\s*(?:0\s*)?(?:تاریخ\s*پذیرش|پذیرش|تاریخ)",
        r"(?:تاریخ\s*پذیرش|پذیرش|تاریخ)\s*[:：]?\s*([0-9]{4}[/-][0-9]{1,2}[/-][0-9]{1,2})",
    ]
    for pat in patterns:
        for m in re.finditer(pat, full, re.I):
            cand = normalize_digits(m.group(1))
            if validate_jalali_date(cand):
                return cand, m.group(0)
    return None, None

def extract_tav_header(full):
    m = re.search(r"([^\n]{1,60}?)-\s*(آقای|آقاي|خانم)\s+([^\n]{1,40}?)-\s*دکتر\s*([0-9]{1,3})\s*[:：]\s*سن", full)
    if not m: return {}
    family = clean_value(m.group(1), 50)
    title = m.group(2)
    given = clean_value(m.group(3), 50)
    age = int(m.group(4))
    sex = "female" if title == "خانم" else "male"
    name = clean_value(f"{given} {family}", 90)
    return {"patient_name":(name,m.group(0)), "sex":(sex,m.group(0)), "age":(age,m.group(0))}

def parse_patient_from_line(line):
    line = compact_name_spacing(line)
    if not PATIENT_LABEL_RE.search(line) and "نام بیمار" not in line:
        return None, None, None
    before = re.split(r"نام\s*بیمار|نام\s*مراجعه|Patient\s*Name", line, flags=re.I)[0]
    before = before.replace("+"," ").replace("،"," ").replace(","," ").strip()
    if "خانم" in before or "آقای" in before or "آقاي" in before:
        sex = "female" if "خانم" in before else "male"
        name = before.replace("خانم","").replace("آقای","").replace("آقاي","")
        name = clean_value(name, 90)
        if not reject_name_candidate(name):
            return name, sex, line
    parts = re.split(r"نام\s*بیمار|نام\s*مراجعه|Patient\s*Name", line, flags=re.I)
    if len(parts) > 1:
        after = re.split(r"تاریخ|سن|شماره|پزشک|کد|پذیرش|Date|Age", parts[1])[0]
        sex = "female" if "خانم" in after else ("male" if ("آقای" in after or "آقاي" in after) else None)
        name = after.replace("خانم","").replace("آقای","").replace("آقاي","")
        name = clean_value(name, 90)
        if not reject_name_candidate(name):
            return name, sex, line
    return None, None, None

def extract_compact_patient_name(full):
    pats = [
        r"(?:[0-9]{1,3}\s*سال\s*)?(خانم|آقای|آقاي)\s*([آ-ی]{2,50}(?:\s*[آ-ی]{2,30}){0,3})\s*[:：]?\s*نام",
        r"([آ-ی]{2,40})\s*(خانم|آقای|آقاي)\s*[:：]?\s*(?:ند\s*برد|نام)",
    ]
    for pat in pats:
        m = re.search(pat, full)
        if not m: continue
        if m.group(1) in ["خانم","آقای","آقاي"]:
            sex = "female" if m.group(1) == "خانم" else "male"
            name = clean_value(m.group(2), 90)
        else:
            sex = "female" if m.group(2) == "خانم" else "male"
            name = clean_value(m.group(1), 90)
        if not reject_name_candidate(name):
            return name, sex, m.group(0)
    return None, None, None

def extract_age(full, top):
    for pat in [r"سن\s*[:：]?\s*([0-9]{1,3})(?:\s*سال)?", r"([0-9]{1,3})\s*سال\s*سن", r"سال\s*([0-9]{1,3})\s*سن", r"([0-9]{1,3})\s*سال"]:
        for m in re.finditer(pat, top):
            src = m.group(0)
            if re.search(r"Below age|Above age|Reference|Adults|Children|years?:", src, re.I):
                continue
            val = int(m.group(1))
            if 0 <= val <= 120:
                return val, src
    return None, None

def extract_doctor_from_lines(lines):
    for line in lines[:25]:
        if PATIENT_LABEL_RE.search(line) or not DOCTOR_LABEL_RE.search(line):
            continue
        pieces = re.split(r"نام\s*پزشک|پزشک\s*معالج|پزشک|Doctor|Physician", line, flags=re.I)
        candidates = []
        if len(pieces) > 1: candidates.append(pieces[-1])
        candidates.append(pieces[0])
        for cand in candidates:
            cand = clean_value(cand.replace(":"," ").replace("："," "), 90)
            if cand and not reject_doctor_candidate(cand):
                return cand, line
    return None, None

def extract_center(lines):
    for l in lines[:12]:
        if any(k in l for k in ["آزمایشگاه","آزمايشگاه","Laboratory","درمانگاه","بیمارستان","Nobin"]):
            if TABLE_TOKENS_RE.search(l): continue
            value = clean_value(l, 140)
            if value: return value, l
    return None, None

def validate_center_value(value):
    if not value: return "missing"
    low = value.lower()
    if low.endswith(" and") or len(value) < 6 or re.search(r"\b(and|specialized and)$", low):
        return "review"
    return "valid"

def extract_common_fields(text):
    full = normalize_text(text)
    lines = [l.strip() for l in full.splitlines() if l.strip()]
    top = "\n".join(lines[:25])
    center, center_src = extract_center(lines)
    nid, nid_src = extract_strict_national_id(full)
    tracking = None; tracking_src = None
    for pat in [r"\b([A-Za-z]-\d{5}-\d{3,})\b", r"(?:شماره\s*پذیرش|پذیرش|Admission|Report\s*No|شماره)\s*[:：]?\s*([A-Za-z0-9_-]{3,})", r"([0-9]{4,8})\s*(?:0\s*)?پذیرش"]:
        m = re.search(pat, full, re.I)
        if m:
            tracking, tracking_src = m.group(1), m.group(0); break
    date, date_src = extract_date(full)
    tav = extract_tav_header(full)
    patient_name = sex = age = None
    patient_src = sex_src = age_src = None
    if "patient_name" in tav: patient_name, patient_src = tav["patient_name"]
    if "sex" in tav: sex, sex_src = tav["sex"]
    if "age" in tav: age, age_src = tav["age"]
    if not patient_name:
        for line in lines[:30]:
            cand, sx, src = parse_patient_from_line(line)
            if cand:
                patient_name, patient_src = cand, src
                if sx and not sex: sex, sex_src = sx, src
                break
    if not patient_name:
        cand, sx, src = extract_compact_patient_name(full)
        if cand:
            patient_name, patient_src = cand, src
            if sx and not sex: sex, sex_src = sx, src
    if not sex and patient_src:
        if "خانم" in patient_src or re.search(r"\bFemale\b", patient_src, re.I):
            sex, sex_src = "female", patient_src
        elif "آقای" in patient_src or "آقاي" in patient_src or re.search(r"\bMale\b", patient_src, re.I):
            sex, sex_src = "male", patient_src
    if age is None:
        age, age_src = extract_age(full, top)
    doctor, doctor_src = extract_doctor_from_lines(lines)
    def field(value, conf, src, extra=None):
        d = {"value":value, "confidence":conf if value not in [None,""] else 0.0, "source_text":sanitize_text(src) if src else None}
        if extra: d.update(extra)
        return d
    return {
        "center_name": field(center, 0.7 if validate_center_value(center)=="valid" else 0.55, center_src, {"center_validation_hint":validate_center_value(center)}),
        "national_id": {"value":hash_value(nid), "confidence":0.9 if nid else 0.0, "source_text":sanitize_text(nid_src) if nid_src else None, "strict_label_required":True},
        "tracking_number": field(tracking,0.85,tracking_src),
        "date_of_test_or_report": field(date,0.9,date_src,{"calendar":"jalali" if date else None}),
        "patient_name": field(patient_name,0.88 if patient_name else 0.0,patient_src),
        "sex": field(sex,0.85 if sex else 0.0,sex_src),
        "age": field(age,0.85 if age is not None else 0.0,age_src),
        "doctor_name": field(doctor,0.75 if doctor else 0.0,doctor_src),
    }


In [10]:
# =========================
# Lab parsing and production gate
# =========================

NUM_RE = re.compile(r"(?<!\d)([*<>]?\d+(?:\.\d+)?)(?!\d)")
UNIT_RE = re.compile(r"(10\^?\s*[36]\s*/?\s*(?:uL|µL|μL|pL)|10\*?[36]\s*/?\s*(?:uL|µL|μL|pL)|mg/dL|mg/dl|g/dL|g/dl|U/L|IU/L|mIU/L|µIU/mL|uIU/mL|ng/mL|ng/ml|pg/ml|IU/L|%|fL|fl|pg|Ratio|Sec|mm/hr|mg/d)", re.I)
METHOD_RE = re.compile(r"\b(Photometr|ELISA|ECLIA|CLIA|EIA|HPLC|FLC|KIA\s*&\s*ELFA)\b", re.I)
GUIDELINE_WORDS_RE = re.compile(r"\b(Normal|Borderline|Bordreline|Borrderline|Desirable|High risk|Low risk|Adults?|Males?|Females?|Women|Men|Pregnant|Below age|Above age|Reference|Range|Prognose|Abnormal|Doubtful)\b", re.I)

def alias_regex(alias): return re.escape(alias).replace(r"\ ", r"\s+")
def line_is_known_test(line):
    clean = line.strip()
    for std, alias in ALL_ALIAS_PHRASES:
        if re.fullmatch(alias_regex(alias), clean, re.I):
            return std, alias
    return None, None
def find_test_occurrences(text):
    t = normalize_text(text)
    for std, alias in ALL_ALIAS_PHRASES:
        pat = re.compile(rf"(?i)(?<![A-Za-z]){alias_regex(alias)}(?![A-Za-z])")
        for m in pat.finditer(t):
            yield {"std":std,"alias":alias,"start":m.start(),"end":m.end()}
def next_test_boundary(text, start, min_ahead=5, max_window=240):
    end_limit = min(len(text), start+max_window)
    next_pos = end_limit
    for occ in find_test_occurrences(text[start+min_ahead:end_limit]):
        pos = start + min_ahead + occ["start"]
        if pos > start + min_ahead:
            next_pos = min(next_pos, pos)
    return next_pos
def parse_reference_range(ref):
    if not ref: return None
    r = str(ref).replace(" ","").replace("–","-")
    m = re.match(r"^<\s*(\d+(?:\.\d+)?)$", r)
    if m: return ("upper", float(m.group(1)))
    m = re.match(r"^>\s*(\d+(?:\.\d+)?)$", r)
    if m: return ("lower", float(m.group(1)))
    m = re.match(r"^(\d+(?:\.\d+)?)-(\d+(?:\.\d+)?)$", r)
    if m:
        a,b = float(m.group(1)), float(m.group(2))
        if a <= b: return ("range",a,b)
    return None
def flag_from_reference(value, ref):
    parsed = parse_reference_range(ref)
    if value is None or not parsed: return None
    if parsed[0] == "upper": return "High" if value > parsed[1] else None
    if parsed[0] == "lower": return "Low" if value < parsed[1] else None
    _, lo, hi = parsed
    if value < lo: return "Low"
    if value > hi: return "High"
    return None
def extract_explicit_flag(block, result_end_pos):
    local = block[result_end_pos: result_end_pos+60]
    local = re.sub(r"(High|Low)\s+risk|High\s*[<>]\s*\d+|Low\s*[<>]\s*\d+", "", local, flags=re.I)
    m = re.search(r"(?<![A-Za-z])(High|Low|Critical)(?![A-Za-z])", local, re.I)
    if m: return m.group(1).capitalize()
    m = re.search(r"(?<![A-Za-z.])(H|L)(?![A-Za-z.])", local)
    if m: return "High" if m.group(1)=="H" else "Low"
    if "*" in block[:result_end_pos+20]: return "rechecked"
    return None
def final_flag(value, ref, explicit_flag):
    if parse_reference_range(ref):
        return flag_from_reference(value, ref)
    return explicit_flag
def infer_section(std):
    if std in {"WBC","RBC","HGB","HCT","MCV","MCH","MCHC","RDW-CV","RDW-SD","PLT","PDW","MPV","NEUT%","LYMPH%","MONO%","EO%","BASO%","ESR"}: return "hematology"
    if std in {"PT","PT Control","INR","PTT"}: return "coagulation"
    if std in {"TSH","T3","T4","Free T4","Vitamin D","Ferritin","CRP"}: return "special_biochemistry_or_hormone"
    if std in {"Urine Culture","Color","Appearance","Specific Gravity","pH","Protein","Urine Glucose","Ketone","Bilirubin Urine","Urobilinogen","Nitrite","Blood/Hb","WBC/HPF","RBC/HPF","Bacteria","Mucus","Casts","Crystals"}: return "urine_or_microbiology"
    return "biochemistry"
def try_decimal_correction(std, value, raw):
    corrections=[]; corrected=value
    if std=="MCH" and value>100: corrected=value/10; corrections.append("decimal_shift_divide_by_10_for_MCH")
    elif std=="HbA1c" and value>30: corrected=value/100 if value>100 else value/10; corrections.append("decimal_shift_for_HbA1c")
    elif std=="WBC" and 40<value<200: corrected=value/10; corrections.append("decimal_shift_divide_by_10_for_WBC")
    elif std=="Ferritin" and value>3000 and "." not in str(raw): corrections.append("possible_decimal_missing_for_Ferritin")
    return corrected, corrections
def validate_lab_row(row):
    std = row.get("test_name_standard"); raw=row.get("result_value"); numeric=row.get("result_numeric")
    reasons=[]; validation_status="valid"; save_allowed=True; corrected_value=None
    if numeric is None:
        qv = re.sub(r"\s+"," ",str(raw or "").strip().lower())
        if std in QUALITATIVE_TESTS and any(qv.startswith(x) or x in qv for x in QUALITATIVE_VALID_VALUES):
            return {"validation_status":"valid","save_allowed":True,"reason_codes":[],"corrected_numeric":None,"corrected_value":None}
        return {"validation_status":"review","save_allowed":False,"reason_codes":["non_numeric_or_unrecognized_textual_result"],"corrected_numeric":None,"corrected_value":None}
    corrected_numeric, corrections = try_decimal_correction(std, float(numeric), str(raw))
    if corrections:
        reasons += corrections
        if std in ROW_SANITY_LIMITS:
            lo,hi=ROW_SANITY_LIMITS[std]
            if lo <= corrected_numeric <= hi:
                validation_status="corrected_review"; save_allowed=False; corrected_value=str(corrected_numeric)
    check = corrected_numeric if corrected_value is not None else float(numeric)
    if std in ROW_SANITY_LIMITS:
        lo,hi=ROW_SANITY_LIMITS[std]
        if not (lo <= check <= hi):
            validation_status="invalid"; save_allowed=False; reasons.append(f"outside_physiologic_range_{lo}_{hi}")
    if row.get("confidence",0) < 0.60:
        if validation_status=="valid": validation_status="review"
        save_allowed=False; reasons.append("low_row_confidence")
    if row.get("unit") is None and std not in QUALITATIVE_TESTS and std not in {"pH"}:
        if validation_status=="valid": validation_status="review"
        save_allowed=False; reasons.append("missing_unit")
    return {"validation_status":validation_status,"save_allowed":save_allowed,"reason_codes":sorted(set(reasons)),"corrected_numeric":corrected_numeric if corrected_value is not None else None,"corrected_value":corrected_value}
def parse_test_block(std, alias, block, confidence_base=0.70):
    b = block.strip()
    b_after = re.sub(rf"(?i)^\s*{alias_regex(alias)}\s*", "", b).strip()
    num_match = None
    for m in NUM_RE.finditer(b_after[:120]):
        before = b_after[max(0,m.start()-30):m.start()]
        if GUIDELINE_WORDS_RE.search(before) and m.start()>25: continue
        if re.search(r"\d{4}[/-]\d", b_after[max(0,m.start()-10):m.end()+12]): continue
        num_match=m; break
    if not num_match: return None
    result_raw = num_match.group(1).lstrip("*")
    try: numeric=float(re.sub(r"^[<>]+","",result_raw))
    except Exception: numeric=None
    after = b_after[num_match.end():]
    unit_m = UNIT_RE.search(after[:70])
    unit = unit_m.group(0) if unit_m else None
    ref=None
    ref_m = re.search(r"([<>]\s*\d+(?:\.\d+)?|\d+(?:\.\d+)?\s*[-–]\s*\d+(?:\.\d+)?)", after[:100])
    if ref_m: ref=re.sub(r"\s+","",ref_m.group(1))
    method_m = METHOD_RE.search(after[:140])
    explicit = extract_explicit_flag(b_after, num_match.end())
    row = {"section":infer_section(std),"test_name_standard":std,"test_name_raw":alias,"result_value":result_raw,"result_numeric":numeric,
           "unit":unit,"reference_range":ref,"flag":final_flag(numeric,ref,explicit),"method":method_m.group(1) if method_m else None,
           "confidence":confidence_base,"source_text":sanitize_text(b[:300])}
    row.update(validate_lab_row(row)); return row
def extract_clean_multiline_blocks(text):
    lines=[l.strip() for l in normalize_text(text).splitlines() if l.strip()]
    rows=[]; i=0
    while i < len(lines):
        std,alias=line_is_known_test(lines[i])
        if not std: i+=1; continue
        j=i+1; block_lines=[lines[i]]
        while j < len(lines):
            nstd,_=line_is_known_test(lines[j])
            if nstd: break
            if not re.fullmatch(r"(Test|Result|Unit|Normal Range|Reference Range|Flag|Method|Biochemistry|Hematology)", lines[j], re.I):
                block_lines.append(lines[j])
            if len(block_lines)>=8: break
            j+=1
        row=parse_test_block(std,alias,"\n".join(block_lines),0.82)
        if row: rows.append(row)
        i=max(j,i+1)
    return rows
def extract_messy_window_rows(text):
    flat=re.sub(r"\s+"," ",normalize_text(text)); rows=[]
    for occ in sorted(find_test_occurrences(flat), key=lambda x:x["start"]):
        start=occ["start"]; end=next_test_boundary(flat,start,max_window=210)
        row=parse_test_block(occ["std"],occ["alias"],flat[start:end],0.58)
        if row: rows.append(row)
    return rows
def extract_urine_culture(text):
    flat=re.sub(r"\s+"," ",normalize_text(text)); rows=[]
    m=re.search(r"(No\s+(?:bacteria\s+)?growth\s+after\s+(?:24|48)\s*(?:hrs?|hours)\.?)", flat, re.I)
    if m:
        row={"section":"microbiology","test_name_standard":"Urine Culture","test_name_raw":"Culture","result_value":m.group(1),"result_numeric":None,"unit":None,"reference_range":None,"flag":None,"method":None,"confidence":0.88,"source_text":sanitize_text(flat[max(0,m.start()-80):m.end()+80])}
        row.update(validate_lab_row(row)); rows.append(row)
    q_patterns={"Color":r"Color\s+(Yellow|Amber|Red|Pale|Straw|Yeon|Yetlow)","Appearance":r"Appearance\s+(Clear|Semi[- ]?Clear|Turbid|Cloudy|Rac)","Protein":r"Protein\s+(Negative|Positive|Trace|[0-9+]+)","Urine Glucose":r"Glucose\s+(Negative|Positive|Trace|[0-9+]+)","Ketone":r"Ketone\s+(Negative|Positive|Trace|[0-9+]+)","Nitrite":r"Nitrite\s+(Negative|Positive)","Bacteria":r"Bacteria\s+(Negative|Not seen|Look at culture|Few|Rare|Many)","Mucus":r"Mucus\s+(Negative|Not seen|Few|Rare|Many)","Casts":r"Casts?\s+(Negative|Not seen|Few|Rare|Many)","Crystals":r"Crystals?\s+(Negative|Not seen|Few|Rare|Many)"}
    for std,pat in q_patterns.items():
        m=re.search(pat,flat,re.I)
        if m:
            row={"section":"urine_analysis","test_name_standard":std,"test_name_raw":std,"result_value":m.group(1),"result_numeric":None,"unit":None,"reference_range":None,"flag":None,"method":None,"confidence":0.66,"source_text":sanitize_text(flat[max(0,m.start()-80):m.end()+80])}
            row.update(validate_lab_row(row)); rows.append(row)
    return rows
def dedup_lab(rows):
    best={}
    for r in rows:
        key=(r.get("test_name_standard"),str(r.get("result_value")).lower(),r.get("unit"))
        if key not in best or r.get("confidence",0)>best[key].get("confidence",0): best[key]=r
    return list(best.values())
def extract_lab_results(text):
    return dedup_lab(extract_clean_multiline_blocks(text)+extract_messy_window_rows(text)+extract_urine_culture(text))


In [11]:
# =========================
# Backend safety policy
# =========================

def validate_common_fields(common, doc_type, backend_patient_context_available=False):
    field_status={}; reasons=[]
    for k,v in common.items():
        value=v.get("value"); conf=v.get("confidence",0)
        if value in [None,""]: field_status[k]="missing"
        elif k=="center_name" and v.get("center_validation_hint")=="review": field_status[k]="review"
        elif conf<0.7: field_status[k]="review"
        else: field_status[k]="valid"
    has=lambda k: common.get(k,{}).get("value") not in [None,""]
    if CONFIG["require_report_date_for_backend_save"] and not has("date_of_test_or_report"): reasons.append("missing_report_date")
    patient_ok = has("patient_name") or has("national_id") or backend_patient_context_available
    if CONFIG["require_patient_or_backend_context_for_backend_save"] and not patient_ok: reasons.append("missing_patient_identity_or_backend_context")
    return {"field_validation_status":field_status,"critical_field_reason_codes":reasons,"critical_fields_valid_for_backend":len(reasons)==0}
def valid_common_payload(common, field_validation_status):
    payload={}
    for k,v in common.items():
        if field_validation_status.get(k)=="valid" and v.get("value") is not None:
            payload[k]={"value":v.get("value"),"confidence":v.get("confidence"),"source_text":v.get("source_text")}
            if "calendar" in v: payload[k]["calendar"]=v.get("calendar")
            if k=="national_id": payload[k]["is_hash"]=True
    return payload
def validate_rows_for_backend(labs):
    counts=Counter(); safe=[]; unsafe=[]
    for r in labs:
        st=r.get("validation_status","review"); counts[st]+=1
        if r.get("save_allowed"): safe.append(r)
        else: unsafe.append({"test_name_standard":r.get("test_name_standard"),"result_value":r.get("result_value"),"validation_status":st,"reason_codes":r.get("reason_codes",[]),"source_text":r.get("source_text")})
    return {"row_validation_status":dict(counts),"safe_row_count":len(safe),"unsafe_row_count":len(unsafe),"unsafe_rows_sample":unsafe[:50],"safe_rows":safe}
def decide_extraction_status(path, quality, chosen, medical_ok, doc_type, common, labs):
    if quality.get("status") in ["poor_quality","unreadable"] and Path(path).suffix.lower()!=".pdf": return "poor_quality_rejected", ["input_quality_too_poor"]
    if not chosen.text.strip(): return "ocr_failed", ["ocr_failed"]
    if chosen.status in ["poor_ocr_text","gibberish_or_bad_layout_text"] and not chosen.score_details.get("culture_signal"): return "ocr_unreliable_needs_reupload", [f"ocr_layout_status={chosen.status}"]
    if not medical_ok: return "unrelated_or_not_medical", ["no_medical_signals"]
    if doc_type=="lab":
        if not labs: return "partial_extraction", ["no_lab_rows_extracted"]
        return "extracted_good", []
    return "needs_manual_review", ["unknown_or_weak_document_type"]
def build_backend_save_policy(extraction_status, common, common_validation, row_validation, doc_type):
    reasons=[]; route="save_to_database"; allowed=True
    if extraction_status in ["poor_quality_rejected","ocr_failed","ocr_unreliable_needs_reupload"]:
        allowed=False; route="ask_user_reupload"; reasons.append(extraction_status)
    if extraction_status=="unrelated_or_not_medical":
        allowed=False; route="reject_unrelated"; reasons.append("unrelated_or_not_medical")
    if extraction_status in ["partial_extraction","needs_manual_review"]:
        allowed=False; route="manual_review"; reasons.append(extraction_status)
    if not common_validation["critical_fields_valid_for_backend"]:
        allowed=False
        if route=="save_to_database": route="manual_review"
        reasons += common_validation["critical_field_reason_codes"]
    if row_validation["safe_row_count"]==0 and doc_type=="lab":
        allowed=False
        if route=="save_to_database": route="manual_review"
        reasons.append("no_safe_lab_rows")
    if row_validation["unsafe_row_count"]>0: reasons.append("some_rows_require_review")
    payload={"common_fields":valid_common_payload(common, common_validation["field_validation_status"]),"lab_results":row_validation["safe_rows"]}
    if not allowed: payload={"common_fields":{},"lab_results":[]}
    return {"backend_save_allowed":allowed,"review_required":route=="manual_review","reupload_required":route=="ask_user_reupload","route":route,"reason_codes":sorted(set(reasons)),"safe_payload":payload}
def common_to_parts(common, field_validation_status):
    return [{"part_type":"common_field","section":"header","field_name":k,"value":v.get("value"),"confidence":v.get("confidence",0),"source_text":v.get("source_text"),"field_validation_status":field_validation_status.get(k,"unknown")} for k,v in common.items() if v.get("value") is not None]
def lab_to_parts(labs):
    rows=[]
    for r in labs:
        rows.append({"part_type":"lab_result","section":r.get("section"),"field_name":r.get("test_name_standard"),"value":r.get("result_value"),"result_numeric":r.get("result_numeric"),"unit":r.get("unit"),"reference_range":r.get("reference_range"),"flag":r.get("flag"),"method":r.get("method"),"confidence":r.get("confidence"),"source_text":r.get("source_text"),"test_name_raw":r.get("test_name_raw"),"row_validation_status":r.get("validation_status"),"row_save_allowed":r.get("save_allowed"),"row_reason_codes":"|".join(r.get("reason_codes",[])),"corrected_numeric":r.get("corrected_numeric"),"corrected_value":r.get("corrected_value")})
    return rows


In [12]:
# =========================
# Batch OCR/Layout + extraction
# =========================

candidate_rows=[]; words_rows=[]; lines_rows=[]; sections_rows=[]
summary_rows=[]; parts_rows=[]; json_rows=[]; text_rows=[]
documents=[]; backend_safe_payloads=[]; review_queue=[]; reupload_required=[]

for index, path in enumerate(FILES, 1):
    sid=f"{index:04d}_{safe_id(path)}"
    print(f"[v9 {index}/{len(FILES)}] {path.name}")
    try:
        quality=assess_file_quality(path)
        source_preview=create_source_preview(path,sid)
        candidates, pdf_meta = get_ocr_candidates_for_file(path,sid)
        chosen=select_best_candidate(candidates)
        text=normalize_text(chosen.text)
        for c in candidates:
            row={"index":index,"filename":path.name,"source_relative_path":relpath(path),"candidate_id":c.candidate_id,"page_number":c.page_number,"variant_name":c.variant_name,"psm":c.psm,"lang":c.lang,"confidence":c.confidence,"layout_score":c.score,"layout_status":c.status,"text_length":len(c.text),"error":c.error,"is_chosen":c.candidate_id==chosen.candidate_id}
            for k,v in c.score_details.items(): row[f"score_{k}"]=v
            candidate_rows.append(row)
        wr,lr,sr=make_words_lines_sections_tables(index,path.name,chosen,relpath(path))
        words_rows+=wr; lines_rows+=lr; sections_rows+=sr
        medical_ok, med_hits, relevance_score = is_medical_text(text)
        doc_type, doc_conf, doc_signals = classify_document(text)
        common=extract_common_fields(text)
        labs=extract_lab_results(text) if doc_type=="lab" or medical_ok else []
        extraction_status,warnings=decide_extraction_status(path,quality,chosen,medical_ok,doc_type,common,labs)
        common_validation=validate_common_fields(common,doc_type,CONFIG["backend_patient_context_available"])
        row_validation=validate_rows_for_backend(labs)
        save_policy=build_backend_save_policy(extraction_status,common,common_validation,row_validation,doc_type)
        parts=common_to_parts(common,common_validation["field_validation_status"])+lab_to_parts(labs)
        text_path=TEXT_DIR/f"{sid}.txt"; san_text_path=SANITIZED_TEXT_DIR/f"{sid}.txt"; json_path=JSON_DIR/f"{sid}.json"
        text_path.write_text(text,encoding="utf-8"); san_text_path.write_text(sanitize_text(text),encoding="utf-8")
        doc_json={"document":{"index":index,"filename":path.name,"source_relative_path":relpath(path),"file_ext":path.suffix.lower(),"mime_type":file_mime_guess(path),"file_size_kb":round(path.stat().st_size/1024,2),"extraction_status":extraction_status,"warnings":warnings,"document_type":doc_type,"document_type_confidence":doc_conf,"relevance_score":relevance_score,"medical_hits":med_hits,"source_preview":source_preview},
                  "quality":quality,
                  "ocr":{"chosen_candidate_id":chosen.candidate_id,"success":bool(text.strip()),"confidence":chosen.confidence,"text_length":len(text),"variant_name":chosen.variant_name,"psm":chosen.psm,"lang":chosen.lang,"layout_score":chosen.score,"layout_status":chosen.status,"score_details":chosen.score_details},
                  "common_fields":common,"field_validation":common_validation,"row_validation":{k:v for k,v in row_validation.items() if k!="safe_rows"},"backend_save_policy":save_policy,"backend_safe_payload":save_policy["safe_payload"],"all_extracted_lab_results_for_review":labs,"extracted_parts":parts,"extracted_text":text}
        json_path.write_text(json.dumps(doc_json,ensure_ascii=False,indent=2),encoding="utf-8")
        documents.append(doc_json)
        if save_policy["backend_save_allowed"]: backend_safe_payloads.append(doc_json["backend_safe_payload"])
        elif save_policy["route"]=="manual_review": review_queue.append(doc_json)
        elif save_policy["route"]=="ask_user_reupload": reupload_required.append(doc_json)
        for r in parts:
            rr=dict(r); rr.update({"index":index,"filename":path.name,"extraction_status":extraction_status,"backend_save_allowed":save_policy["backend_save_allowed"],"backend_route":save_policy["route"],"document_type":doc_type,"source_relative_path":relpath(path),"text_relative_path":relpath(text_path),"json_relative_path":relpath(json_path)})
            parts_rows.append(rr)
        summary_rows.append({"index":index,"filename":path.name,"source_relative_path":relpath(path),"extraction_status":extraction_status,"backend_save_allowed":save_policy["backend_save_allowed"],"review_required":save_policy["review_required"],"reupload_required":save_policy["reupload_required"],"backend_route":save_policy["route"],"backend_reason_codes":"|".join(save_policy["reason_codes"]),"warnings":";".join(warnings),"quality_status":quality.get("status"),"quality_score":quality.get("overall_quality_score"),"quality_issues":";".join(quality.get("issues",[])),"ocr_success":bool(text.strip()),"ocr_confidence":round(float(chosen.confidence or 0),3),"ocr_text_length":len(text),"ocr_layout_status":chosen.status,"ocr_layout_score":round(float(chosen.score or 0),3),"chosen_variant":chosen.variant_name,"chosen_psm":chosen.psm,"chosen_lang":chosen.lang,"document_type":doc_type,"document_type_confidence":doc_conf,"common_field_count":sum(1 for v in common.values() if v.get("value") is not None),"critical_fields_valid_for_backend":common_validation["critical_fields_valid_for_backend"],"field_validation_status":json.dumps(common_validation["field_validation_status"],ensure_ascii=False),"row_validation_status":json.dumps(row_validation["row_validation_status"],ensure_ascii=False),"safe_row_count":row_validation["safe_row_count"],"unsafe_row_count":row_validation["unsafe_row_count"],"lab_result_count":len(labs),"patient_name_found":common.get("patient_name",{}).get("value") is not None,"date_found":common.get("date_of_test_or_report",{}).get("value") is not None,"age_found":common.get("age",{}).get("value") is not None,"sex_found":common.get("sex",{}).get("value") is not None,"national_id_hash_found":common.get("national_id",{}).get("value") is not None,"text_relative_path":relpath(text_path),"json_relative_path":relpath(json_path),"source_preview":source_preview})
        json_rows.append({"index":index,"filename":path.name,"extraction_status":extraction_status,"backend_save_allowed":save_policy["backend_save_allowed"],"backend_route":save_policy["route"],"document_type":doc_type,"quality_json":json.dumps(quality,ensure_ascii=False),"ocr_json":json.dumps(doc_json["ocr"],ensure_ascii=False),"common_fields_json":json.dumps(common,ensure_ascii=False),"field_validation_json":json.dumps(common_validation,ensure_ascii=False),"row_validation_json":json.dumps({k:v for k,v in row_validation.items() if k!="safe_rows"},ensure_ascii=False),"backend_save_policy_json":json.dumps(save_policy,ensure_ascii=False),"backend_safe_payload_json":json.dumps(doc_json["backend_safe_payload"],ensure_ascii=False),"all_extracted_lab_results_for_review_json":json.dumps(labs,ensure_ascii=False),"extracted_parts_json":json.dumps(parts,ensure_ascii=False),"document_json":json.dumps(doc_json,ensure_ascii=False),"source_relative_path":relpath(path),"text_relative_path":relpath(text_path),"json_relative_path":relpath(json_path)})
        text_rows.append({"index":index,"filename":path.name,"source_relative_path":relpath(path),"text_relative_path":relpath(text_path),"sanitized_text_relative_path":relpath(san_text_path),"extracted_text":text,"sanitized_text":sanitize_text(text)})
    except Exception as e:
        traceback.print_exc()
        summary_rows.append({"index":index,"filename":path.name,"source_relative_path":relpath(path),"extraction_status":"notebook_error","backend_save_allowed":False,"review_required":True,"reupload_required":False,"backend_route":"manual_review","backend_reason_codes":str(e)})

summary_df=pd.DataFrame(summary_rows); parts_df=pd.DataFrame(parts_rows); json_rows_df=pd.DataFrame(json_rows); texts_df=pd.DataFrame(text_rows)
candidates_df=pd.DataFrame(candidate_rows); words_df=pd.DataFrame(words_rows); lines_df=pd.DataFrame(lines_rows); sections_df=pd.DataFrame(sections_rows)

paths={"document_summary.csv":OUTPUT_DIR/"document_summary.csv","extracted_parts.csv":OUTPUT_DIR/"extracted_parts.csv","document_json_rows.csv":OUTPUT_DIR/"document_json_rows.csv","extracted_texts.csv":OUTPUT_DIR/"extracted_texts.csv","ocr_candidates.csv":OUTPUT_DIR/"ocr_candidates.csv","ocr_words.csv":OUTPUT_DIR/"ocr_words.csv","ocr_lines.csv":OUTPUT_DIR/"ocr_lines.csv","ocr_sections.csv":OUTPUT_DIR/"ocr_sections.csv","documents.jsonl":OUTPUT_DIR/"documents.jsonl","backend_safe_payloads.jsonl":OUTPUT_DIR/"backend_safe_payloads.jsonl","review_queue.jsonl":OUTPUT_DIR/"review_queue.jsonl","reupload_required.jsonl":OUTPUT_DIR/"reupload_required.jsonl"}

summary_df.to_csv(paths["document_summary.csv"],index=False,encoding="utf-8-sig")
parts_df.to_csv(paths["extracted_parts.csv"],index=False,encoding="utf-8-sig")
json_rows_df.to_csv(paths["document_json_rows.csv"],index=False,encoding="utf-8-sig")
texts_df.to_csv(paths["extracted_texts.csv"],index=False,encoding="utf-8-sig")
candidates_df.to_csv(paths["ocr_candidates.csv"],index=False,encoding="utf-8-sig")
words_df.to_csv(paths["ocr_words.csv"],index=False,encoding="utf-8-sig")
lines_df.to_csv(paths["ocr_lines.csv"],index=False,encoding="utf-8-sig")
sections_df.to_csv(paths["ocr_sections.csv"],index=False,encoding="utf-8-sig")
for key,data in [("documents.jsonl",documents),("backend_safe_payloads.jsonl",backend_safe_payloads),("review_queue.jsonl",review_queue),("reupload_required.jsonl",reupload_required)]:
    with paths[key].open("w",encoding="utf-8") as f:
        for d in data: f.write(json.dumps(d,ensure_ascii=False)+"\n")

print("Saved outputs:")
for name,p in paths.items(): print("-",relpath(p))
print("\nExtraction status counts:"); display(summary_df["extraction_status"].value_counts().to_frame("count"))
print("\nBackend route counts:"); display(summary_df["backend_route"].value_counts().to_frame("count"))
print("\nOCR layout status counts:"); display(summary_df["ocr_layout_status"].value_counts().to_frame("count"))
display(summary_df); display(candidates_df.sort_values(["index","layout_score"],ascending=[True,False]).head(80)); display(lines_df.head(120)); display(parts_df.head(150))


[v9 1/20] -2147483648_-210193.jpg
[v9 2/20] -2147483648_-210195.jpg
[v9 3/20] 0014161672_14041209_O_00404121721.pdf
[v9 4/20] 0020139871_14041209_O_00404121714.pdf
[v9 5/20] 0021858845_14041209_O_00404121731.pdf
[v9 6/20] 0023471026_14041209_O_00404121728.pdf
[v9 7/20] 0024150010_14041209_O_00404121722.pdf
[v9 8/20] 0025631314_14041209_O_00404121726.pdf
[v9 9/20] 0025692283_14041209_O_00404121730.pdf
[v9 10/20] 20260427_181554.jpg
[v9 11/20] 20260427_181636.jpg
[v9 12/20] 20260427_181654.jpg
[v9 13/20] 20260427_181713.jpg
[v9 14/20] 20260427_181919.jpg
[v9 15/20] ۲۰۲۶۰۴۲۵_۲۲۰۷۰۰.jpg
[v9 16/20] ۲۰۲۶۰۴۲۸_۱۵۰۰۳۶.jpg
[v9 17/20] ۲۰۲۶۰۴۲۸_۱۵۰۰۵۱.jpg
[v9 18/20] ۲۰۲۶۰۴۲۸_۱۵۰۱۰۶.jpg
[v9 19/20] ۲۰۲۶۰۴۲۸_۱۵۰۱۲۰.jpg
[v9 20/20] ۲۰۲۶۰۴۲۸_۱۵۰۲۲۸.jpg
Saved outputs:
- notebook_outputs/selected_samples_v9_ocr_layout/document_summary.csv
- notebook_outputs/selected_samples_v9_ocr_layout/extracted_parts.csv
- notebook_outputs/selected_samples_v9_ocr_layout/document_json_rows.csv
- notebook_outputs/selecte

,count
extraction_status,
extracted_good,16
ocr_unreliable_needs_reupload,4



Backend route counts:


,count
backend_route,
manual_review,9
save_to_database,7
ask_user_reupload,4



OCR layout status counts:


,count
ocr_layout_status,
good_layout_text,10
gibberish_or_bad_layout_text,4
usable_short_culture_text,3
usable_noisy_layout_text,3


,index,filename,source_relative_path,extraction_status,backend_save_allowed,review_required,reupload_required,backend_route,backend_reason_codes,warnings,...,unsafe_row_count,lab_result_count,patient_name_found,date_found,age_found,sex_found,national_id_hash_found,text_relative_path,json_relative_path,source_preview
0,1,-2147483648_-210193.jpg,samples/selected samples/-2147483648_-210193.jpg,extracted_good,False,True,False,manual_review,missing_patient_identity_or_backend_context|mi...,,...,11,22,False,False,False,False,False,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...
1,2,-2147483648_-210195.jpg,samples/selected samples/-2147483648_-210195.jpg,ocr_unreliable_needs_reupload,False,False,True,ask_user_reupload,no_safe_lab_rows|ocr_unreliable_needs_reupload,ocr_layout_status=gibberish_or_bad_layout_text,...,0,0,True,True,False,True,False,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...
2,3,0014161672_14041209_O_00404121721.pdf,samples/selected samples/0014161672_14041209_O...,extracted_good,True,False,False,save_to_database,some_rows_require_review,,...,3,22,True,True,True,True,True,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...
3,4,0020139871_14041209_O_00404121714.pdf,samples/selected samples/0020139871_14041209_O...,extracted_good,True,False,False,save_to_database,some_rows_require_review,,...,1,4,True,True,True,True,True,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...
4,5,0021858845_14041209_O_00404121731.pdf,samples/selected samples/0021858845_14041209_O...,extracted_good,True,False,False,save_to_database,some_rows_require_review,,...,2,17,True,True,True,True,True,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...
5,6,0023471026_14041209_O_00404121728.pdf,samples/selected samples/0023471026_14041209_O...,extracted_good,True,False,False,save_to_database,some_rows_require_review,,...,1,2,True,True,True,True,True,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...
6,7,0024150010_14041209_O_00404121722.pdf,samples/selected samples/0024150010_14041209_O...,extracted_good,True,False,False,save_to_database,some_rows_require_review,,...,3,22,True,True,True,True,True,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...
7,8,0025631314_14041209_O_00404121726.pdf,samples/selected samples/0025631314_14041209_O...,extracted_good,True,False,False,save_to_database,some_rows_require_review,,...,1,4,True,True,True,True,True,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...
8,9,0025692283_14041209_O_00404121730.pdf,samples/selected samples/0025692283_14041209_O...,extracted_good,True,False,False,save_to_database,some_rows_require_review,,...,1,4,True,True,True,True,True,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...
9,10,20260427_181554.jpg,samples/selected samples/20260427_181554.jpg,extracted_good,False,True,False,manual_review,missing_patient_identity_or_backend_context|mi...,,...,13,13,False,False,False,False,False,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...


,index,filename,source_relative_path,candidate_id,page_number,variant_name,psm,lang,confidence,layout_score,...,score_numeric_hits,score_table_patterns,score_common_field_signals,score_line_count,score_gibberish_ratio,score_culture_signal,score_impossible_value_penalty,score_avg_word_confidence,score_avg_line_confidence,score_pdf_meta
19,1,-2147483648_-210193.jpg,samples/selected samples/-2147483648_-210193.jpg,0001_-2147483648_-210193_p1_upscale3_original_...,1,upscale3_original,6.0,eng,0.744103,0.856800,...,100,23,0,80,0.035,True,0,0.744,0.724,NaN
18,1,-2147483648_-210193.jpg,samples/selected samples/-2147483648_-210193.jpg,0001_-2147483648_-210193_p1_upscale3_original_...,1,upscale3_original,6.0,eng+fas,0.760724,0.856315,...,97,22,0,80,0.031,True,0,0.761,0.741,NaN
10,1,-2147483648_-210193.jpg,samples/selected samples/-2147483648_-210193.jpg,0001_-2147483648_-210193_p1_upscale2_original_...,1,upscale2_original,6.0,eng+fas,0.712903,0.835483,...,92,23,0,77,0.027,True,0,0.713,0.694,NaN
35,1,-2147483648_-210193.jpg,samples/selected samples/-2147483648_-210193.jpg,0001_-2147483648_-210193_p1_gray_autocontrast_...,1,gray_autocontrast_upscale2,6.0,eng,0.715504,0.835230,...,91,22,0,77,0.028,True,0,0.716,0.696,NaN
11,1,-2147483648_-210193.jpg,samples/selected samples/-2147483648_-210193.jpg,0001_-2147483648_-210193_p1_upscale2_original_...,1,upscale2_original,6.0,eng,0.702842,0.832216,...,90,22,0,77,0.027,True,0,0.703,0.686,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27,1,-2147483648_-210193.jpg,samples/selected samples/-2147483648_-210193.jpg,0001_-2147483648_-210193_p1_gray_autocontrast_...,1,gray_autocontrast,6.0,eng,0.325286,0.317448,...,31,3,0,68,0.051,False,0,0.325,0.287,NaN
3,1,-2147483648_-210193.jpg,samples/selected samples/-2147483648_-210193.jpg,0001_-2147483648_-210193_p1_original_oriented_...,1,original_oriented,6.0,eng,0.329761,0.316444,...,30,3,0,68,0.055,False,0,0.330,0.290,NaN
207,1,-2147483648_-210193.jpg,samples/selected samples/-2147483648_-210193.jpg,0001_-2147483648_-210193_p1_shadow_removed_rot...,1,shadow_removed_rot270,12.0,eng,0.455862,0.310247,...,14,2,0,79,0.034,False,0,0.456,0.454,NaN
71,1,-2147483648_-210193.jpg,samples/selected samples/-2147483648_-210193.jpg,0001_-2147483648_-210193_p1_clahe_medium_psm12...,1,clahe_medium,12.0,eng,0.478000,0.305575,...,16,1,0,86,0.064,False,0,0.478,0.448,NaN


,line_id,text,left,top,right,bottom,width,height,word_count,avg_conf,index,filename,candidate_id,page_number,variant_name,psm,lang,section,source_relative_path
0,1,Special Biochemistry,71.0,43,463.0,82,392.0,39.0,2,0.960000,1,-2147483648_-210193.jpg,0001_-2147483648_-210193_p1_upscale3_original_...,1,upscale3_original,6.0,eng,biochemistry,samples/selected samples/-2147483648_-210193.jpg
1,2,Test Result Unit Method Refrence Value,60.0,116,1246.0,154,1186.0,38.0,6,0.830000,1,-2147483648_-210193.jpg,0001_-2147483648_-210193_p1_upscale3_original_...,1,upscale3_original,6.0,eng,biochemistry,samples/selected samples/-2147483648_-210193.jpg
2,3,Normal diabetic ; Up fo 5.0,1079.0,165,1460.0,205,381.0,40.0,6,0.548333,1,-2147483648_-210193.jpg,0001_-2147483648_-210193_p1_upscale3_original_...,1,upscale3_original,6.0,eng,biochemistry,samples/selected samples/-2147483648_-210193.jpg
3,4,Good diahetic control :6-7,1079.0,204,1436.0,246,357.0,42.0,4,0.805000,1,-2147483648_-210193.jpg,0001_-2147483648_-210193_p1_upscale3_original_...,1,upscale3_original,6.0,eng,biochemistry,samples/selected samples/-2147483648_-210193.jpg
4,5,Fair diabetic control :7-9,1079.0,244,1455.0,285,376.0,41.0,4,0.507500,1,-2147483648_-210193.jpg,0001_-2147483648_-210193_p1_upscale3_original_...,1,upscale3_original,6.0,eng,biochemistry,samples/selected samples/-2147483648_-210193.jpg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,6,امینی طھران- آقای کورش- دکتر34 : سن,NaN,5,NaN,5,NaN,NaN,7,0.950000,3,0014161672_14041209_O_00404121721.pdf,0003_0014161672_14041209_O_00404121721_pdf_tex...,1,pdf_text_layer,NaN,embedded,unknown,samples/selected samples/0014161672_14041209_O...
116,7,شماره :,NaN,6,NaN,6,NaN,NaN,2,0.950000,3,0014161672_14041209_O_00404121721.pdf,0003_0014161672_14041209_O_00404121721_pdf_tex...,1,pdf_text_layer,NaN,embedded,unknown,samples/selected samples/0014161672_14041209_O...
117,8,نام :,NaN,7,NaN,7,NaN,NaN,2,0.950000,3,0014161672_14041209_O_00404121721.pdf,0003_0014161672_14041209_O_00404121721_pdf_tex...,1,pdf_text_layer,NaN,embedded,unknown,samples/selected samples/0014161672_14041209_O...
118,9,کد ملی :0014161672,NaN,8,NaN,8,NaN,NaN,3,0.950000,3,0014161672_14041209_O_00404121721.pdf,0003_0014161672_14041209_O_00404121721_pdf_tex...,1,pdf_text_layer,NaN,embedded,unknown,samples/selected samples/0014161672_14041209_O...


,part_type,section,field_name,value,result_numeric,unit,reference_range,flag,method,confidence,...,index,filename,extraction_status,backend_save_allowed,backend_route,document_type,source_relative_path,text_relative_path,json_relative_path,field_validation_status
0,lab_result,biochemistry,EAG,97,97.00,mg/d,<20,High,NaN,0.58,...,1,-2147483648_-210193.jpg,extracted_good,False,manual_review,lab,samples/selected samples/-2147483648_-210193.jpg,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...,NaN
1,lab_result,special_biochemistry_or_hormone,Vitamin D,25,25.00,NaN,NaN,NaN,NaN,0.58,...,1,-2147483648_-210193.jpg,extracted_good,False,manual_review,lab,samples/selected samples/-2147483648_-210193.jpg,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...,NaN
2,lab_result,special_biochemistry_or_hormone,Vitamin D,3,3.00,ng/ml,30-100,Low,CLIA,0.58,...,1,-2147483648_-210193.jpg,extracted_good,False,manual_review,lab,samples/selected samples/-2147483648_-210193.jpg,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...,NaN
3,lab_result,special_biochemistry_or_hormone,CRP,3,3.00,NaN,NaN,Low,NaN,0.58,...,1,-2147483648_-210193.jpg,extracted_good,False,manual_review,lab,samples/selected samples/-2147483648_-210193.jpg,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...,NaN
4,lab_result,special_biochemistry_or_hormone,TSH,2.55,2.55,mIU/L,535-4,Low,ECLIA,0.58,...,1,-2147483648_-210193.jpg,extracted_good,False,manual_review,lab,samples/selected samples/-2147483648_-210193.jpg,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,common_field,header,age,21,NaN,NaN,NaN,NaN,NaN,0.85,...,9,0025692283_14041209_O_00404121730.pdf,extracted_good,True,save_to_database,lab,samples/selected samples/0025692283_14041209_O...,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...,valid
146,lab_result,biochemistry,FBS,95,95.00,mg/dL,70-115,NaN,Photometr,0.82,...,9,0025692283_14041209_O_00404121730.pdf,extracted_good,True,save_to_database,lab,samples/selected samples/0025692283_14041209_O...,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...,NaN
147,lab_result,biochemistry,AST,14,14.00,U/L,<31,NaN,Photometr,0.82,...,9,0025692283_14041209_O_00404121730.pdf,extracted_good,True,save_to_database,lab,samples/selected samples/0025692283_14041209_O...,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...,NaN
148,lab_result,biochemistry,ALT,15,15.00,U/L,<32,NaN,Photometr,0.82,...,9,0025692283_14041209_O_00404121730.pdf,extracted_good,True,save_to_database,lab,samples/selected samples/0025692283_14041209_O...,notebook_outputs/selected_samples_v9_ocr_layou...,notebook_outputs/selected_samples_v9_ocr_layou...,NaN


In [13]:
# =========================
# HTML OCR/Layout Review Report
# =========================

def df_to_html_table(df, columns=None, max_rows=120):
    if df is None or df.empty:
        return "<p><em>No rows.</em></p>"
    x=df.copy()
    if columns:
        x=x[[c for c in columns if c in x.columns]]
    if len(x)>max_rows:
        x=x.head(max_rows)
    return x.to_html(index=False,escape=True)

sections_html=[]
for d in documents:
    idx=d["document"]["index"]; fn=d["document"]["filename"]; sp=d["document"].get("source_preview"); policy=d["backend_save_policy"]
    preview_html=f'<img src="../../{html.escape(sp)}" style="max-width:700px;border:1px solid #ddd"/>' if sp else ""
    bg="#e8f5e9" if policy["backend_save_allowed"] else ("#fff8e1" if policy["route"]=="manual_review" else "#ffebee")
    cand_sub=candidates_df[candidates_df["index"]==idx].sort_values("layout_score",ascending=False).head(10)
    line_sub=lines_df[lines_df["index"]==idx].head(160)
    psub=parts_df[parts_df["index"]==idx].copy()
    common=psub[psub["part_type"]=="common_field"]
    labs=psub[psub["part_type"]=="lab_result"]
    text=d.get("extracted_text","")
    sections_html.append(
        f'<section style="background:{bg}; padding:16px; margin:20px 0; border-radius:8px;">'
        f'<h2>{idx}. {html.escape(fn)}</h2>'
        f'<p><b>Extraction status:</b> {html.escape(d["document"]["extraction_status"])} | '
        f'<b>Backend route:</b> {html.escape(policy["route"])} | '
        f'<b>Save allowed:</b> {policy["backend_save_allowed"]} | '
        f'<b>Reasons:</b> {html.escape(", ".join(policy["reason_codes"]))}</p>'
        f'<p><b>Chosen OCR:</b> {html.escape(str(d["ocr"]["variant_name"]))}, '
        f'PSM={html.escape(str(d["ocr"]["psm"]))}, lang={html.escape(str(d["ocr"]["lang"]))} | '
        f'<b>Layout status:</b> {html.escape(str(d["ocr"]["layout_status"]))} | '
        f'<b>Layout score:</b> {html.escape(str(round(d["ocr"]["layout_score"], 3)))}</p>'
        f'<details open><summary><b>Source preview</b></summary>{preview_html}</details>'
        f'<details open><summary><b>Top OCR candidates</b></summary>{df_to_html_table(cand_sub, ["candidate_id","variant_name","psm","lang","confidence","layout_score","layout_status","text_length","score_alias_hits","score_table_patterns","score_common_field_signals","score_impossible_value_penalty"])}</details>'
        f'<details open><summary><b>Chosen visual lines</b></summary>{df_to_html_table(line_sub, ["line_id","section","text","avg_conf","left","top","width","height"], 180)}</details>'
        f'<details open><summary><b>Common fields</b></summary>{df_to_html_table(common, ["field_name","value","confidence","field_validation_status","source_text"])}</details>'
        f'<details open><summary><b>Lab rows</b></summary>{df_to_html_table(labs, ["field_name","value","unit","reference_range","flag","confidence","row_validation_status","row_save_allowed","row_reason_codes","corrected_value","source_text"], 220)}</details>'
        f'<details><summary><b>Chosen OCR text</b></summary><pre style="white-space:pre-wrap; background:white; padding:10px; border:1px solid #ddd;">{html.escape(text[:9000])}</pre></details>'
        f'</section>'
    )

html_parts=[
    "<!doctype html><html><head><meta charset='utf-8'/>",
    "<title>Medical OCR/Layout Review v9</title>",
    "<style>",
    "body { font-family: Arial, sans-serif; margin: 24px; line-height: 1.4; }",
    "table { border-collapse: collapse; font-size: 13px; width: 100%; background:white; }",
    "td, th { border: 1px solid #ddd; padding: 6px; vertical-align: top; }",
    "th { background: #f2f2f2; }",
    "pre { direction: ltr; }",
    "</style></head><body>",
    "<h1>Medical OCR/Layout Review v9</h1>",
    "<p>This report is for OCR/layout debugging. Backend should save only from backend_safe_payloads.jsonl.</p>",
    "<h2>Summary</h2>",
    summary_df.to_html(index=False, escape=True),
    *sections_html,
    "</body></html>"
]
review_path=OUTPUT_DIR/"review_ocr_layout.html"
review_path.write_text("\n".join(html_parts),encoding="utf-8")
print("Saved:",relpath(review_path))


Saved: notebook_outputs/selected_samples_v9_ocr_layout/review_ocr_layout.html


In [14]:
# =========================
# QA Views
# =========================

print("Files requiring review or reupload:")
display(summary_df[summary_df["backend_save_allowed"] == False][[
    "index","filename","extraction_status","backend_route","backend_reason_codes",
    "ocr_layout_status","ocr_layout_score","chosen_variant","chosen_psm","chosen_lang",
    "safe_row_count","unsafe_row_count","patient_name_found","date_found","json_relative_path"
]])

print("\nTop OCR candidates per document:")
display(candidates_df.sort_values(["index","layout_score"],ascending=[True,False]).groupby("index").head(5)[[
    "index","filename","variant_name","psm","lang","confidence","layout_score","layout_status",
    "score_alias_hits","score_table_patterns","score_common_field_signals","score_impossible_value_penalty","is_chosen"
]])

print("\nBlocked rows:")
if not parts_df.empty and "row_save_allowed" in parts_df.columns:
    display(parts_df[(parts_df["part_type"]=="lab_result") & (parts_df["row_save_allowed"] == False)][[
        "index","filename","field_name","value","unit","reference_range",
        "row_validation_status","row_reason_codes","corrected_value","source_text"
    ]].head(250))


Files requiring review or reupload:


,index,filename,extraction_status,backend_route,backend_reason_codes,ocr_layout_status,ocr_layout_score,chosen_variant,chosen_psm,chosen_lang,safe_row_count,unsafe_row_count,patient_name_found,date_found,json_relative_path
0,1,-2147483648_-210193.jpg,extracted_good,manual_review,missing_patient_identity_or_backend_context|mi...,usable_short_culture_text,0.857,upscale3_original,6.0,eng,11,11,False,False,notebook_outputs/selected_samples_v9_ocr_layou...
1,2,-2147483648_-210195.jpg,ocr_unreliable_needs_reupload,ask_user_reupload,no_safe_lab_rows|ocr_unreliable_needs_reupload,gibberish_or_bad_layout_text,0.255,gray_autocontrast_upscale2_rot90,12.0,eng+fas,0,0,True,True,notebook_outputs/selected_samples_v9_ocr_layou...
9,10,20260427_181554.jpg,extracted_good,manual_review,missing_patient_identity_or_backend_context|mi...,usable_noisy_layout_text,0.478,shadow_removed_rot90,6.0,eng+fas,0,13,False,False,notebook_outputs/selected_samples_v9_ocr_layou...
10,11,20260427_181636.jpg,ocr_unreliable_needs_reupload,ask_user_reupload,missing_patient_identity_or_backend_context|mi...,gibberish_or_bad_layout_text,0.279,shadow_removed_rot180,12.0,eng+fas,0,4,False,False,notebook_outputs/selected_samples_v9_ocr_layou...
11,12,20260427_181654.jpg,extracted_good,manual_review,missing_patient_identity_or_backend_context|mi...,usable_noisy_layout_text,0.422,threshold_fallback,12.0,eng+fas,2,0,False,False,notebook_outputs/selected_samples_v9_ocr_layou...
12,13,20260427_181713.jpg,extracted_good,manual_review,missing_patient_identity_or_backend_context|mi...,usable_short_culture_text,0.420,shadow_removed_rot90,12.0,eng+fas,1,1,False,False,notebook_outputs/selected_samples_v9_ocr_layou...
13,14,20260427_181919.jpg,extracted_good,manual_review,missing_patient_identity_or_backend_context|mi...,usable_noisy_layout_text,0.533,shadow_removed_rot90,4.0,eng+fas,0,10,False,False,notebook_outputs/selected_samples_v9_ocr_layou...
14,15,۲۰۲۶۰۴۲۵_۲۲۰۷۰۰.jpg,extracted_good,manual_review,no_safe_lab_rows|some_rows_require_review,good_layout_text,0.856,gray_autocontrast,4.0,eng+fas,0,23,True,True,notebook_outputs/selected_samples_v9_ocr_layou...
15,16,۲۰۲۶۰۴۲۸_۱۵۰۰۳۶.jpg,extracted_good,manual_review,missing_patient_identity_or_backend_context|mi...,good_layout_text,0.809,gray_autocontrast_upscale2,4.0,eng+fas,0,20,False,False,notebook_outputs/selected_samples_v9_ocr_layou...
16,17,۲۰۲۶۰۴۲۸_۱۵۰۰۵۱.jpg,ocr_unreliable_needs_reupload,ask_user_reupload,missing_patient_identity_or_backend_context|mi...,gibberish_or_bad_layout_text,0.232,shadow_removed,6.0,eng+fas,0,0,False,False,notebook_outputs/selected_samples_v9_ocr_layou...



Top OCR candidates per document:


,index,filename,variant_name,psm,lang,confidence,layout_score,layout_status,score_alias_hits,score_table_patterns,score_common_field_signals,score_impossible_value_penalty,is_chosen
19,1,-2147483648_-210193.jpg,upscale3_original,6.0,eng,0.744103,0.856800,usable_short_culture_text,36,23,0,0,True
18,1,-2147483648_-210193.jpg,upscale3_original,6.0,eng+fas,0.760724,0.856315,usable_short_culture_text,36,22,0,0,False
10,1,-2147483648_-210193.jpg,upscale2_original,6.0,eng+fas,0.712903,0.835483,usable_short_culture_text,35,23,0,0,False
35,1,-2147483648_-210193.jpg,gray_autocontrast_upscale2,6.0,eng,0.715504,0.835230,usable_short_culture_text,36,22,0,0,False
11,1,-2147483648_-210193.jpg,upscale2_original,6.0,eng,0.702842,0.832216,usable_short_culture_text,35,22,0,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2633,20,۲۰۲۶۰۴۲۸_۱۵۰۲۲۸.jpg,gray_autocontrast_upscale2_rot90,6.0,eng+fas,0.383198,0.377075,gibberish_or_bad_layout_text,7,9,0,0,True
2635,20,۲۰۲۶۰۴۲۸_۱۵۰۲۲۸.jpg,gray_autocontrast_upscale2_rot90,11.0,eng+fas,0.441068,0.330827,gibberish_or_bad_layout_text,7,2,0,0,False
2573,20,۲۰۲۶۰۴۲۸_۱۵۰۲۲۸.jpg,shadow_removed,12.0,eng+fas,0.469675,0.322822,gibberish_or_bad_layout_text,7,2,0,0,False
2693,20,۲۰۲۶۰۴۲۸_۱۵۰۲۲۸.jpg,shadow_removed_rot180,12.0,eng+fas,0.486419,0.299239,gibberish_or_bad_layout_text,6,1,0,0,False



Blocked rows:


,index,filename,field_name,value,unit,reference_range,row_validation_status,row_reason_codes,corrected_value,source_text
0,1,-2147483648_-210193.jpg,EAG,97,mg/d,<20,review,low_row_confidence,NaN,"EAG) 97 mg/dh Defictent:<20 , Jnsuffictent:20-29"
1,1,-2147483648_-210193.jpg,Vitamin D,25,NaN,NaN,review,low_row_confidence|missing_unit,NaN,25-Hydroxy
2,1,-2147483648_-210193.jpg,Vitamin D,3,ng/ml,30-100,review,low_row_confidence,NaN,Vitamin D3 » 15.39 ng/ml CLIA ry Su/fficient:3...
3,1,-2147483648_-210193.jpg,CRP,3,NaN,NaN,review,low_row_confidence|missing_unit,NaN,CRP (Quantitative) *s.3° mg/L Negative: Upto 6...
4,1,-2147483648_-210193.jpg,TSH,2.55,mIU/L,535-4,review,low_row_confidence,NaN,"TSH 2.55 mIU/L ~—s«ECLIA-— 535-4,.78 0, Immuno..."
...,...,...,...,...,...,...,...,...,...,...
265,20,۲۰۲۶۰۴۲۸_۱۵۰۲۲۸.jpg,PTT,3,NaN,NaN,review,low_row_confidence|missing_unit,NaN,"ptt #3 game 2028"" 08:40 3 ae , Fat Fit 15 وه‎ ..."
266,20,۲۰۲۶۰۴۲۸_۱۵۰۲۲۸.jpg,T3,0,NaN,NaN,review,low_row_confidence|missing_unit,NaN,"t3 ) وخ‎ 0 ی 1 7 gf? 4 gone 45 ایب‎ RB 4, و‎ 3..."
267,20,۲۰۲۶۰۴۲۸_۱۵۰۲۲۸.jpg,MCHC,4.997,NaN,NaN,invalid,low_row_confidence|missing_unit|outside_physio...,NaN,MCHC z4.997¢d ys Noe Ps ee 1 i ر‎ Re 6 ee
268,20,۲۰۲۶۰۴۲۸_۱۵۰۲۲۸.jpg,PLT,264,NaN,NaN,review,low_row_confidence|missing_unit,NaN,PLT 264 وم«‎ اد A 2 Te OF A we : “at 7 Pat ge”...
